# Reefer Energy Consumption Forecasting — ARIMAX vs XGBoost

Research internship at **LIS (Laboratoire d'Informatique et des Systèmes)**, TNTM project (CMA CGM × LIS).

**Goal:** forecast the electrical consumption of refrigerated containers (reefers) measured at on-board power
points ("sources"), on a reference voyage (**Leg 30**, Singapore → Valencia, CMA CGM Grace Bay, 2 h time step).

**Notebook structure** (mirrors the internship report):

1. Setup
2. Data loading and dataset construction
3. Leg 30 description and source characteristics
4. ARIMAX — one-shot prediction (V0), multi-horizon study, rolling update (V2), coefficient analysis
5. XGBoost — pooled training and enrichment with contextual features (setpoint, equipment age)

**Evaluation protocol:** chronological 80 % train / 10 % validation / 10 % test split. All model choices
(ARIMAX order, XGBoost hyperparameters, window size) are selected on the validation set only.
Metric: RMSE and relative RMSE (RMSE / mean real consumption on the test period).

> ⚠️ The raw data (CMA CGM) is confidential and is **not** included in this repository.

## 1. Setup

In [2]:
import itertools
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler, RobustScaler, StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# Folder containing the (confidential, not versioned) data — see data/README.md
DATA_DIR = "../data"

## 2. Data loading and dataset construction

The final dataset merges, for each leg:
- the **consumption** of each power source (minute data, resampled to 2 h),
- the **ambient air temperature** (and relative wind speed),
- the **reefers loaded** on the leg (position Bay/Tier/Row, setpoint, age), linked to each source through its physical position.

Filters: only **DECK** positions with at least 3 reefers; sources whose consumption stays below 1 kW for ≥ 95 % of the leg are dropped.

### 2.1 Legs

In [3]:
def get_all_leg():
    df_legs = pd.read_excel(os.path.join(DATA_DIR, os.path.join(DATA_DIR, "GRACE_BAY_Environmental_Operationnal_Data.xlsx")),sheet_name='Legs')
    return df_legs

In [4]:
def get_leg_by_date(df, start_date, end_date):
    df = df.copy()

    
    df["Departure time"] = pd.to_datetime(df["Departure time"])
    df["Arrival Time"] = pd.to_datetime(df["Arrival Time"])

   
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # Keep rows matching the interval
    result = df[
        (df["Departure time"] == start_date) &
        (df["Arrival Time"] == end_date)
    ]

    return result

In [5]:
def filter_legs(df_legs, selected_legs=None):
    """
    Filter legs list to focus on subset of legs.

    Parameters
    ----------
    df_legs : DataFrame
      

    selected_legs : list
        Leg numbers to keep. 

    
    """

    if selected_legs is None:
        selected_legs = {4, 10, 15, 16, 21, 22, 23, 24, 30, 34, 44}

    df_legs = df_legs.copy()
    df_legs["Leg"] = df_legs["Leg"].astype(int)

    df_legs=df_legs[df_legs["Leg"].isin(selected_legs)].copy()
    
    
    # dealing with combined leg
   
    leg_15 = df_legs[df_legs["Leg"] == 15].iloc[0]
    leg_16 = df_legs[df_legs["Leg"] == 16].iloc[0]
    merged_row = leg_15.copy()

    
    merged_row["Departure time"] = leg_15["Departure time"]
    merged_row["Departure port"] = leg_15["Departure port"]
    merged_row["Departure port code"] = leg_15["Departure port code"]

   
    merged_row["Arrival Time"] = leg_16["Arrival Time"]
    merged_row["Arrival port"] = leg_16["Arrival port"]
    merged_row["Arrival port code"] = leg_16["Arrival port code"]

   
    merged_row["Leg"] = leg_15["Leg"]

   
    merged_row["reefer_20"] = leg_15["reefer_20"] 
    merged_row["reefer_40"] = leg_15["reefer_40"] 

    
    df_without_15_16 = df_legs[~df_legs["Leg"].isin([15, 16])].copy()

    
    df_result = pd.concat(
        [df_without_15_16, merged_row.to_frame().T],
     ignore_index=True
    )

    return df_result

### 2.2 Environmental data (ambient temperature, wind)

In [6]:
def get_AmbientTemperature(startDate, endDate):

    df = pd.read_excel(os.path.join(DATA_DIR, "GRACE_BAY_Environmental_Operationnal_Data.xlsx"),sheet_name='environmental_factors')

    
    df["TS"] = pd.to_datetime(df['TS'], errors='coerce', dayfirst=True)
    df['AIR_TEMPERATURE_2M'] = pd.to_numeric(df['AIR_TEMPERATURE_2M'], errors='coerce')
   

   
    df['AIR_TEMPERATURE_2M'] = df['AIR_TEMPERATURE_2M'].replace(["Unknown", "unknown"], np.nan)
    
     # Reindex with start and end
    ts_range = pd.DatetimeIndex([startDate, endDate])
    df_interp = (
    df.set_index("TS")
      .reindex(pd.concat([pd.Series(df["TS"]), pd.Series(ts_range)]))
      .sort_index()
      .interpolate(method="time")
)

    # Filter 
    df_filtred = df_interp.loc[startDate:endDate]
    
    df_temp_hourly = df_filtred.resample("2h").mean(numeric_only=True)
    return df_temp_hourly
   

In [7]:
def get_WindSpeed(startDate, endDate):
    """
    Load the relative wind speed for a date range, resampled to 2 h.

    Parameters
    ----------
    startDate : str
        Start date, format 'YYYY-MM-DD HH:MM:SS'
    endDate : str
        End date, format 'YYYY-MM-DD HH:MM:SS'

    Returns
    -------
    df_wind_2h : DataFrame
        DataFrame indexed by timestamp (2 h frequency),
        with the column 'Relative Wind Speed' (kn)
    """

  
    df = pd.read_csv(os.path.join(DATA_DIR, "Grace Bay Wind.csv"))

    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce', dayfirst=True)

    df = df.rename(columns={
        'Relative Wind Speed [SMARTSHIP] (kn)': 'Relative Wind Speed'
    })

    df['Relative Wind Speed'] = pd.to_numeric(df['Relative Wind Speed'], errors='coerce')
    df['Relative Wind Speed'] = df['Relative Wind Speed'].replace(["Unknown", "unknown"], np.nan)
        

    ts_range = pd.DatetimeIndex([startDate, endDate])
    timestamps = df["TIMESTAMP"].copy()
    df_interp = (
        df.set_index("TIMESTAMP")
          .reindex(pd.concat([pd.Series(df["TIMESTAMP"]), pd.Series(ts_range)]))
          .sort_index()
          .interpolate(method="time")
    )

    df_filtred = df_interp.loc[startDate:endDate]
    df_wind_2h = df_filtred.resample("2h").mean(numeric_only=True)
    return df_wind_2h

### 2.3 Loaded reefers and their positions

In [8]:
def get_load_data(start_date, end_date):
  
    df = pd.read_excel(os.path.join(DATA_DIR, "GRACE_BAY_Environmental_Operationnal_Data.xlsx"), sheet_name="operational_factors")

    
    df["VCDHD_BAPLIE_INTEGRATION_DATE"] = pd.to_datetime(
        df["VCDHD_BAPLIE_INTEGRATION_DATE"]    )

    
    start_date = pd.to_datetime(start_date).date()
    end_date = pd.to_datetime(end_date).date()

    print("start_date =", start_date)
    print("end_date   =", end_date) 
    
    filtered_df = df[
        (df["VCDHD_BAPLIE_INTEGRATION_DATE"].dt.date >= start_date) &
        (df["VCDHD_BAPLIE_INTEGRATION_DATE"].dt.date <= end_date)
    ]
  

    return filtered_df
    

In [9]:
def determine_cote(stow_number):
    # Convert to string to extract the middle digits
    stow_str = str(stow_number)
   #len stow number = 6
    if len(stow_str) >= 6:
        middle_digits = int(stow_str[2:4])  # Extract the 2 middle digits
        # Check whether the digits are even or odd
        if middle_digits % 2 == 0:
            return 'PS'  # Pair
        else:
            return 'STD'   # Impair
    else: # len stow number < 6
        middle_digits = int(stow_str[1:3])  # Extract the 2 middle digits
        # Check whether the digits are even or odd
        if middle_digits % 2 == 0:
            return 'PS'  # Pair
        else:
            return 'STD'   # Impair
    #return None  # malformed stow number

def determine_pos(stow_number):
    # Convert to string to extract the middle digits
    stow_str = str(stow_number)
    if len(stow_str) >= 6:
        middle_digits = int(stow_str[4:6])  # Extract the 2 middle digits
        
        # Check whether the digits are even or odd
        if middle_digits <= 22:
            return 'CH'  # cale
        else:
            return 'DECK'   # ponte
    #return None  # malformed stow number
    else:
         middle_digits = int(stow_str[3:5])
         if middle_digits <= 22:
            return 'CH'  # cale
         else:
            return 'DECK'   # ponte
def determine_bay(stow_number):
    # Convert to string to extract the middle digits
    stow_str = str(stow_number)
    if len(stow_str) >= 6:
        bay = int(stow_str[0:2])  # Extract the first 2 digits 
    else:
         bay = int(stow_str[0:1])  # Extract the first digit
    return bay    

In [10]:
def get_reefers_data(df):
  """
    Mapping slot or stowNumber to BAY / TIER/ROW format

    Parameters
    ----------
    df: DataFrame
        DataFrame containing columns 'VCDHD_BAPLIE_INTEGRATION_DATE', 'SLOT', 'TEMPERATURE_CARRIAGE'.

    Returns
    -------
    
        A dataframe with the following added columns:
        - Bay 
        - Row 
        - Tier 
        - nb_reefer
       

   
"""
  df_reef=df.copy()
  df_reef['Row'] = df_reef['SLOT'].apply(determine_cote)
  df_reef['Tier'] = df_reef['SLOT'].apply(determine_pos)
  df_reef['Bay'] = df_reef['SLOT'].apply(determine_bay)
  
  return df_reef


 

In [11]:
def get_nbReefer_by_position_and_setpoint(df_reefer):
    """
     calculate the number of reefer for each (setpoint, bay, tier,row) 
    
    Parameters
    ----------
    df_reefer: DataFrame
        DataFrame containing columns 'Bay', 'Tier', 'Row', 'SLOT', 'TEMPERATURE_CARRIAGE'.

    Returns
    -------
    
        A dataframe with the following  columns:
        - Bay 
        - Row 
        - Tier
        - Setpoint 
        - nb_reefer
          
    """
    df_reef=df_reefer.copy()
    reefer_per_position = (
    df_reef.groupby(["TEMPERATURE_CARRIAGE", "Row", "Tier", "Bay"])
      .size()
      .reset_index(name="nb_reefer")
        )
    reefer_per_position =reefer_per_position .rename(columns={
        "TEMPERATURE_CARRIAGE": "Setpoint"
    })
    
    return reefer_per_position

In [12]:
def get_nbReefer_by_position(df_reefer):
    """
     calculate the number of reefer for each position (bay, tier,row) 
    
    Parameters
    ----------
    df_reefer: DataFrame
        DataFrame containing columns 'Bay', 'Tier', 'Row', 'SLOT', 'TEMPERATURE_CARRIAGE'.

    Returns
    -------
    
        A dataframe with the following  columns:
        - Bay 
        - Row 
        - Tier
        - Setpoint 
        - nb_reefer
          
    """
    df_reef=df_reefer.copy()
    reefer_per_position = (
    df_reef.groupby(["Row", "Tier", "Bay"])
      .size()
      .reset_index(name="nb_reefer")
        )
   
    
    return reefer_per_position

In [13]:
def filter_df_reefer(df_reefer, min_nb_reefer=3):
    """
    Filter  df_reefer to keep only rows located in DECK positions and nb_reefer >= min.
    """

    df_reef = df_reefer.copy()

    reefer_per_position = get_nbReefer_by_position(df_reef)

    valid_positions = reefer_per_position[
        (reefer_per_position["nb_reefer"] >= min_nb_reefer)
        & (reefer_per_position["Tier"] == "DECK")
    ]

    df_filtered = df_reef.merge(
        valid_positions[["Row", "Tier", "Bay", "nb_reefer"]],
        on=["Row", "Tier", "Bay"],
        how="inner"
    )

    return df_filtered

### 2.4 Consumption per source

In [14]:
_CONS_CACHE = {}


def _load_consumption_file(filename):
    """Read and parse a consumption CSV once, then reuse it (the files are large)."""
    if filename not in _CONS_CACHE:
        _CONS_CACHE.clear()  # keep a single leg in memory
        df = pd.read_csv(os.path.join(DATA_DIR, filename), sep=';', low_memory=False)
        df['Value'] = df['Value'].astype(str).str.replace(',', '.').astype(float)
        df['TimestampUTC'] = pd.to_datetime(df['TimestampUTC'])
        _CONS_CACHE[filename] = df
    return _CONS_CACHE[filename]


def SQLtoDataframe(RP, start, end, leg_num=30):
    leg_files = {
        21: 'cons/consLeg21.csv',
        22: 'cons/consLeg22.csv',
        23: 'cons/consLeg23.csv',
        24: 'cons/consLeg24.csv',
        30: 'cons/consLeg30_new.csv',
        34: 'cons/consLeg34.csv',
        44: 'cons/consLeg44.csv',
    }
    filename = leg_files.get(leg_num, 'cons/consLeg30_new.csv')
    df = _load_consumption_file(filename)

    # Filter by source
    df = df[df['SourceId'] == int(RP)]
    
    # Filter by dates
    df = df[(df['TimestampUTC'] >= pd.to_datetime(start)) &
            (df['TimestampUTC'] <= pd.to_datetime(end))]
    
    return df[['ID', 'Value', 'SourceId', 'TimestampUTC']]

In [15]:
def getSourceList():
    df = pd.read_csv(os.path.join(DATA_DIR, "sourcesReefers.csv"), sep=';')
    return df[['ID', 'Name']]

In [16]:
def get_consumption_leg(df_source,start,end,value_threshold=1, ratio_threshold=0.95, leg_num=30):
    
    df_source["ID"]=df_source["ID"].astype(str)
    all_data = []
    
    for pos_id in df_source["ID"]:
        df = SQLtoDataframe(pos_id,start,end,leg_num=leg_num)  
        df = df.copy()
        all_data.append(df)
    
    # Concatenate everything into one dataframe
    df_cons_leg = pd.concat(all_data, ignore_index=True)
    
    # Detect positions with all-zero values
    unused_positions = (
        df_cons_leg.groupby("SourceId")["Value"]
        #.apply(lambda x: (x < 1).all())
        .apply(lambda x: (x < value_threshold).mean() >= ratio_threshold)
    )
    unused_positions = unused_positions[unused_positions].index
  
     # Drop them
    df= df_cons_leg[~df_cons_leg["SourceId"].isin(unused_positions)]
    # convert into 2h
    df_2h = ( df
    .set_index("TimestampUTC")
    .groupby("SourceId")
    .resample("2h")["Value"]
    .mean()
    .reset_index()
    )
    return df_2h
    

In [17]:
def build_df_consumption(df_sources, df_values):
    #df_sources : all sources ; df_values : df of all sources consumption of a leg
    df_sources["ID"] = df_sources["ID"].astype(int)
    df_values["SourceId"] = df_values["SourceId"].astype(int)
    # Merge using SourceId (left) and ID (right)
    df_merged = df_values.merge(
        df_sources,
        left_on="SourceId",
        right_on="ID",
        how="left"
    )

    # Rename columns
    df_merged = df_merged.rename(columns={
        "Name": "SourceName",
        "Value": "Value",
        "SourceId": "SourceId",
        "TimestampUTC": "TimestampUTC"
    })

    # Select final structure
    df_final = df_merged[["TimestampUTC", "Value", "SourceId", "SourceName"]]

    return df_final

In [18]:
def display_consumption(df_cons):
    for name, group in df_cons.groupby("SourceName"):

        fig = px.line(
        group,
        x="TimestampUTC",
        y="Value",
        title=f"Consumption - {name}"
    )

        fig.show()

### 2.5 Final dataset (consumption + temperature + reefer data)

In [19]:
def extract_position_from_sourceName(name):
    """
    Extract one or more (Bay, Tier, Row) tuples from a source name.

   

    Parameters
    ----------
    name : SourceName .

    Returns
    -------
    list of tuples (Bay, Tier, Row) .
    """ 
    if pd.isna(name):
        return []

    source_name = str(name)

    results = []

    # Pattern 1: deck
    pattern_deck = re.findall(r'Bay_(\d+)_Deck_([A-Za-z]+)', source_name, flags=re.IGNORECASE)
    for bay, row in pattern_deck:
        results.append((int(bay), "DECK", row.upper()))

    # Pattern 2: CH
    pattern_ch = re.findall(r'Bay_(\d+)_CH\d*_([A-Za-z]+)', source_name, flags=re.IGNORECASE)
    for bay, row in pattern_ch:
        results.append((int(bay), "CH", row.upper()))
    
    # Pattern 3: Bay_70_and_74_Deck_PS
    pattern_double_deck = re.findall(
        r'Bay_(\d+)_and_(\d+)_Deck_([A-Za-z]+)',
        source_name,
        flags=re.IGNORECASE
    )
    for bay1, bay2, row in pattern_double_deck:
        results.append((int(bay1), "DECK", row.upper()))
        results.append((int(bay2), "DECK", row.upper()))

    # Remove duplicates while preserving order
    seen = set()
    unique_results = []
    for item in results:
        if item not in seen:
            seen.add(item)
            unique_results.append(item)

    return unique_results

In [20]:
def build_final_df(df_cons, df_temp, df_reefers , df_wind=None):
    """
    Build the final dataframe by merging consumption, temperature, and reefer_data.

    
    Parameters
    ----------
    df_cons : DataFrame contains columns: ['TimestampUTC', 'Value', 'SourceId', 'SourceName']

    df_temp : DataFrame contains column: ['AIR_TEMPERATURE_2M']

    df_reefers : DataFrame contains columns : ['VCDHD_BAPLIE_INTEGRATION_DATE', 'SLOT', 'Bay', 'Tier', 'Row']

    df_wind : DataFrame contains columns: ['TimestampUTC', 'Relative Wind Speed']

    Returns
    -------
    pandas.DataFrame
        Final merged dataframe.
    """

    # -----------------------------
    # 1. Prepare copies
    # -----------------------------
    df_cons_copy = df_cons.copy()
    df_temp_copy = df_temp.copy()
    df_reefers_copy = df_reefers.copy()
    df_wind_copy = df_wind.copy() if df_wind is not None else None

    # -----------------------------
    # 2. Normalize timestamps
    # -----------------------------
    

    # df_temp has timestamp as index, rename the index column to TimestampUTC
    df_temp_copy.index = pd.to_datetime(df_temp_copy.index, errors="coerce")
    df_temp_copy = df_temp_copy.reset_index()

    # rename the index column to TimestampUTC
    if "index" in df_temp_copy.columns:
        df_temp_copy = df_temp_copy.rename(columns={"index": "TimestampUTC"})
    elif df_temp_copy.columns[0] != "TimestampUTC":
        df_temp_copy = df_temp_copy.rename(columns={df_temp.columns[0]: "TimestampUTC"})

   
    print (df_temp_copy.columns)
    # -----------------------------
    # 3. Merge consumption with temperature
    # -----------------------------
    df_merge_1 = df_cons_copy.merge(
        df_temp_copy[["TimestampUTC", "AIR_TEMPERATURE_2M"]],
        on="TimestampUTC",
        how="left"
    )
    
    # -----------------------------
    # 3bis. Merge with wind speed 
    # -----------------------------
    if df_wind is not None:

        df_wind_copy = df_wind.copy()

        df_wind_copy.index = pd.to_datetime(df_wind_copy.index, errors="coerce")
        df_wind_copy = df_wind_copy.reset_index()

        df_wind_copy = df_wind_copy.rename(columns={df_wind_copy.columns[0]: "TimestampUTC"})

        df_wind_copy["TimestampUTC"] = pd.to_datetime(df_wind_copy["TimestampUTC"], errors="coerce")

        df_merge_1 = df_merge_1.merge(
            df_wind_copy[["TimestampUTC", "Relative Wind Speed"]],
            on="TimestampUTC",
            how="left"
        )

    # -----------------------------
    # 4. Extract positions from SourceName
    # -----------------------------
    df_merge_1["positions"] = df_merge_1["SourceName"].apply(extract_position_from_sourceName)

    # Keep only rows with at least one extracted position
    df_merge_1 = df_merge_1[df_merge_1["positions"].map(len) > 0].copy()

    # Explode because one source can map to multiple bays
    df_merge_1 = df_merge_1.explode("positions")

    # Split tuple into Bay, Tier, Row
    df_merge_1[["Bay", "Tier", "Row"]] = pd.DataFrame(
        df_merge_1["positions"].tolist(),
        index=df_merge_1.index
    )

    df_merge_1 = df_merge_1.drop(columns=["positions"])

    # -----------------------------
    # 5. Normalize reefer keys
    # -----------------------------
    df_reefers_copy["Bay"] = pd.to_numeric(df_reefers_copy["Bay"], errors="coerce")
    df_reefers_copy["Tier"] = df_reefers_copy["Tier"].astype(str).str.upper().str.strip()
    df_reefers_copy["Row"] = df_reefers_copy["Row"].astype(str).str.upper().str.strip()

    df_merge_1["Bay"] = pd.to_numeric(df_merge_1["Bay"], errors="coerce")
    df_merge_1["Tier"] = df_merge_1["Tier"].astype(str).str.upper().str.strip()
    df_merge_1["Row"] = df_merge_1["Row"].astype(str).str.upper().str.strip()

    # Optional: normalize common spelling
    df_reefers_copy["Row"] = df_reefers_copy["Row"].replace({"STD": "STD", "PS": "PS"})
    df_merge_1["Row"] = df_merge_1["Row"].replace({"STD": "STD", "PS": "PS"})

    # -----------------------------
    # 6. Merge with reefer positions
    # -----------------------------
    df_final = df_merge_1.merge(
        df_reefers_copy,
        on=["Bay", "Tier", "Row"],
        how="inner"
    )

    return df_final

In [21]:
def build_final_df_for_one_leg(
    leg_info,
    df_source=None,
    min_nb_reefer=3,
    verbose=False
):
    """
    Build df_final for one leg.

    Parameters
    ----------
    leg_info : pandas Series
        

    df_source : DataFrame, 

    min_nb_reefer : int, default=3 (for filtering)
       

   

    verbose : bool, default=False
        If True, print intermediate information.

    Returns
    -------
    DataFrame
        Final dataframe for the given leg.
    """

    # -----------------------------
    # 1. Get leg dates
    # -----------------------------
    #startDate = leg_info["Departure time"]
    #endDate = leg_info["Arrival Time"]
    
    startDate = pd.to_datetime(leg_info["Departure time"]).strftime("%Y-%m-%d %H:%M:%S")
    endDate = pd.to_datetime(leg_info["Arrival Time"]).strftime("%Y-%m-%d %H:%M:%S")

    leg_id = leg_info["Leg"]

    if verbose:
        print(f"\n========== Processing Leg {leg_id} ==========")
        print("Start:", startDate)
        print("End:", endDate)

    # -----------------------------
    # 2. Get ambient temperature
    # -----------------------------
    df_temp = get_AmbientTemperature(startDate, endDate)
    df_wind = get_WindSpeed(startDate, endDate)

    if verbose:
        print("**** Ambient temperature ****")
        print(df_temp.head())

    # -----------------------------
    # 3. Get loaded reefers
    # -----------------------------
    df_reefers = get_load_data(startDate, endDate)
    
    if verbose:
        print("**** Loaded reefers ****")
        print(df_reefers)

    # -----------------------------
    # 4. Add position info
    # -----------------------------
    df_reefers_data = get_reefers_data(df_reefers)

    if verbose:
        print("**** Reefers with positions ****")
        print(df_reefers_data)

    # -----------------------------
    # 5. Filter reefers
    # -----------------------------
    reefer_filtered = filter_df_reefer(
        df_reefers_data,
        min_nb_reefer=min_nb_reefer
        
    )

    if verbose:
        print("**** Reefer positions after filtering ****")
        print(get_nbReefer_by_position(reefer_filtered))

    # -----------------------------
    # 6. Get consumption data
    # -----------------------------
    if df_source is None:
        df_source = getSourceList()

    df_consumption = get_consumption_leg(df_source, startDate, endDate, leg_num=int(leg_id))

    df_cons_leg = build_df_consumption(df_source, df_consumption)

    if verbose:
        print("**** Consumption data ****")
        print(df_cons_leg)

    # -----------------------------
    # 7. Build final dataframe
    # -----------------------------
    df_final = build_final_df(df_cons_leg, df_temp, reefer_filtered, df_wind=df_wind)

    # -----------------------------
    # 8. Add leg metadata
    # -----------------------------
    df_final["Leg"] = leg_id
    df_final["Departure time"] = startDate
    df_final["Arrival Time"] = endDate

    df_final["Departure port"] = leg_info["Departure port"]
    df_final["Departure port code"] = leg_info["Departure port code"]

    df_final["Arrival port"] = leg_info["Arrival port"]
    df_final["Arrival port code"] = leg_info["Arrival port code"]

    if "reefer_20" in leg_info.index:
        df_final["leg_reefer_20"] = leg_info["reefer_20"]

    if "reefer_40" in leg_info.index:
        df_final["leg_reefer_40"] = leg_info["reefer_40"]

    if verbose:
        print("**** df_final ****")
        print(df_final)
       

    return df_final

In [22]:
def build_final_df_for_all_legs(
    df_legs,
    min_nb_reefer=3,
    verbose=False
):
    """
    Build one final dataframe for all legs.

    Parameters
    ----------
    df_legs : DataFrame
        

    min_nb_reefer : int, default=3
       

    
    verbose : bool, default=False
        If True, print intermediate information.

    Returns
    -------
    DataFrame
        Concatenated final dataframe for all legs.
    """

    all_final_dfs = []

    # Load source list once instead of inside each loop
    df_source = getSourceList()

    for idx, leg_row in df_legs.iterrows():

        try:
            df_final_leg = build_final_df_for_one_leg(
                leg_row,
                df_source
                
            )

            all_final_dfs.append(df_final_leg)

        except Exception as e:
            print(f"Error while processing Leg {leg_row['Leg']}: {e}")

    if len(all_final_dfs) == 0:
        return pd.DataFrame()

    df_final_all_legs = pd.concat(all_final_dfs, ignore_index=True)

    return df_final_all_legs

Build the dataset for the 7 legs used in this study (21, 22, 23, 24, 30, 34, 44).

In [ ]:
# main: all legs 


df_legs=get_all_leg()
print("**** all legs *******")
print(df_legs)
print("****************")
print("*******long legs *********")
df_long_legs=filter_legs(df_legs)
print(df_long_legs)
print("****************")

df_final_all = build_final_df_for_all_legs(
    df_legs=df_long_legs,
    min_nb_reefer=3,
    verbose=True
)

print("---------------------------------")
print(df_final_all)

**** all legs *******
        Departure time   Departure port Departure port code  \
0  2024-05-26 17:18:00          NINGBO                CNNGB   
1  2024-05-30 14:48:00         SHANGHAI               CNSHA   
2  2024-06-02 11:30:00         YANTIAN                CNYTN   
3  2024-06-09 00:48:00       SINGAPORE                SGSIN   
4  2024-07-03 01:30:00       TANGER MED               MAPTM   
5  2024-07-08 20:36:00        LE HAVRE                FRLEH   
6  2024-07-14 17:30:00         HAMBURG                DEHAM   
7  2024-07-18 20:36:00           GDANSK               PLGDN   
8  2024-07-24 04:12:00        ROTTERDAM               NLRTM   
9  2024-07-29 22:00:00        ALGECIRAS               ESALG   
10 2024-08-26 15:00:00       PORT KLANG               MYPKG   
11 2024-09-02 16:54:00          NINGBO                CNNGB   
12 2024-09-05 07:36:00        SHANGHAI                CNSHA   
13 2024-09-10 19:06:00         YANTIAN                CNYTN   
14 2024-09-15 01:12:00       SING

## 3. Leg 30 — description and source characteristics

Leg duration, number of sources and reefers, train/validation/test sizes, then per-source statistics
(number of reefers, mean, coefficient of variation, skewness) used to explain ARIMAX performance.

In [ ]:
# ============================================================
# Leg 30 characteristics + summary of both split protocols
# ============================================================

leg_num = 30
df_leg30 = df_final_all[df_final_all["Leg"] == leg_num]

# ---------------- Total leg duration (days) ----------------
row_leg = df_legs[df_legs["Leg"] == leg_num].iloc[0]
dep_time = pd.to_datetime(row_leg["Departure time"])
arr_time = pd.to_datetime(row_leg["Arrival Time"])
duree_heures = (arr_time - dep_time).total_seconds() / 3600
duree_jours = round(duree_heures / 24, 2)

# ---------------- Total number of sources ----------------
n_sources = df_leg30["SourceId"].nunique()

# ---------------- Total number of reefers (sum over all sources of the leg) ----------------
n_reefers_total = (
    df_leg30
    .groupby("SourceId")["nb_reefer"]
    .first()
    .sum()
)

# ---------------- n_obs train/val/test for each protocol ----------------
def compute_split_sizes(df, train_size, val_size, split_mode):
    tailles_train, tailles_val, tailles_test = [], [], []

    for source_id, group in df.groupby("SourceId"):
        n = len(group)
        if split_mode == "train_val_test":
            train_end = int(n * train_size)
            val_end   = int(n * (train_size + val_size))
            tailles_train.append(train_end)
            tailles_val.append(val_end - train_end)
            tailles_test.append(n - val_end)
        elif split_mode == "train_test":
            train_end = int(n * train_size)
            tailles_train.append(train_end)
            tailles_test.append(n - train_end)

    resultat = {
        "n_obs_train_moyen": round(np.mean(tailles_train), 1),
        "n_obs_train_total": int(np.sum(tailles_train)),
        "n_obs_test_moyen":  round(np.mean(tailles_test), 1),
        "n_obs_test_total":  int(np.sum(tailles_test)),
    }
    if split_mode == "train_val_test":
        resultat["n_obs_val_moyen"] = round(np.mean(tailles_val), 1)
        resultat["n_obs_val_total"] = int(np.sum(tailles_val))

    return resultat

# 90/10 protocol (train_test) — uses df_leg30 (already filtered on Leg 30)
splits_90_10 = compute_split_sizes(df_leg30, train_size=0.90, val_size=0.0, split_mode="train_test")

# Protocole 80/10/10 (train_val_test)
splits_80_10_10 = compute_split_sizes(df_leg30, train_size=0.8, val_size=0.10, split_mode="train_val_test")

# ---------------- Final summary table ----------------
df_recap = pd.DataFrame({
    "Protocole": ["90/10 (train/test)", "80/10/10 (train/val/test)"],
    "n_obs_train (total, 12 sources)": [splits_90_10["n_obs_train_total"], splits_80_10_10["n_obs_train_total"]],
    "n_obs_val (total, 12 sources)":   ["—", splits_80_10_10["n_obs_val_total"]],
    "n_obs_test (total, 12 sources)":  [splits_90_10["n_obs_test_total"], splits_80_10_10["n_obs_test_total"]],
})

print("=" * 100)
print(f"LEG {leg_num} CHARACTERISTICS")
print("=" * 100)
print(f"Total leg duration       : {duree_jours} days ({round(duree_heures,1)} hours)")
print(f"Total number of sources  : {n_sources}")
print(f"Total number of reefers  : {n_reefers_total}")
print(f"Resampling frequency     : 2h")
print()
print("Train/validation/test sizes per protocol:")
print(df_recap.to_string(index=False))

LEG 30 CHARACTERISTICS
Total leg duration       : 24.66 days (591.9 hours)
Total number of sources  : 12
Total number of reefers  : 159
Resampling frequency     : 2h

Train/validation/test sizes per protocol:
                Protocole  n_obs_train (total, 12 sources) n_obs_val (total, 12 sources)  n_obs_test (total, 12 sources)
       90/10 (train/test)                            42496                             —                            4727
80/10/10 (train/val/test)                            37774                          4722                            4727


In [ ]:
# ============================================================
# Per-source characteristics — Leg 30 — train/validation/test protocol (80/10/10)
# ============================================================
df_leg30 = df_final_all[df_final_all["Leg"] == 30]

# Reefer info per source (Leg 30)
df_reefer_info = (
    df_leg30
    .groupby(['SourceId', 'SourceName', 'Bay', 'Tier', 'Row'])['nb_reefer']
    .first()
    .reset_index()
)

# Consumption statistics
df_stats = df_leg30.groupby("SourceId").agg(
    n_obs      = ("Value", "count"),
    conso_mean = ("Value", "mean"),
    conso_std  = ("Value", "std"),
    conso_min  = ("Value", "min"),
    conso_max  = ("Value", "max"),
    conso_skew = ("Value", lambda x: x.skew()),
    temp_mean  = ("AIR_TEMPERATURE_2M", "mean"),
).reset_index()

# Fusion stats + reefer info
df_carac = df_reefer_info.merge(df_stats, on="SourceId", how="inner")


print("=" * 100)
print("CHARACTERISTICS PER SOURCE — LEG 30 — train/validation/test protocol (80/10/10)")
print("=" * 100)
print(df_carac.round(2).to_string(index=False))

CHARACTERISTICS PER SOURCE — LEG 30 — train/validation/test protocol (80/10/10)
 SourceId                                             SourceName  Bay Tier Row  nb_reefer  n_obs  conso_mean  conso_std  conso_min  conso_max  conso_skew  temp_mean
      103  RSB1_Switchboard.RSB1-S2_Reefer_RP7_-_Bay_22_Deck_Std   22 DECK STD         10   2970       23.65       3.15      15.65      29.48       -0.61      23.71
      116  RSB2_Switchboard.RSB2-S4_Reefer_RP10_-_Bay_22_Deck_PS   22 DECK  PS          7   2079       17.92       2.98       7.09      22.04       -1.79      23.71
      126 RSB3_Switchboard.RSB3-S4_Reefer_RP21_-_Bay_46_deck_std   46 DECK STD          4   1188       15.02       1.95       7.72      20.02       -0.57      23.71
      127 RSB3_Switchboard.RSB3-S4_Reefer_RP25_-_Bay_50_deck_Std   50 DECK STD         17   5049       61.40       6.19      46.52      72.02       -0.58      23.71
      128 RSB3_Switchboard.RSB3-S4_Reefer_RP29_-_Bay_54_Deck_Std   54 DECK STD         25   742

## 4. ARIMAX

$$\hat{y}(t) = c + \sum_{i=1}^{p} \phi_i\, y(t-i) + \sum_{j=1}^{q} \theta_j\, \varepsilon(t-j) + \beta\, T_{amb}(t)$$

One model per source. The order $(p, q)$ is selected by grid search ($p, q \in \{0,1,2\}$, $d = 0$) on the validation RMSE.

### 4.1 Core functions

In [ ]:
def _tronquer_predictions_equite(df_preds, horizons_fixes):
    """
    Fairness between horizons: truncates the predictions to the same
    number of points for every horizon (the most restrictive one).
    """
    n_points_dispo = {
        h: df_preds[df_preds["horizon_hours"] == h].groupby("SourceId").size().min()
        for h in horizons_fixes
        if not df_preds[df_preds["horizon_hours"] == h].empty
    }
    if not n_points_dispo:
        return df_preds.iloc[0:0], 0
    n_points_cible = min(n_points_dispo.values())

    frames = []
    for source_id, df_source in df_preds.groupby("SourceId"):
        for h in horizons_fixes:
            df_h = df_source[df_source["horizon_hours"] == h].sort_values("TimestampUTC")
            if len(df_h) == 0:
                continue
            frames.append(df_h.iloc[:n_points_cible])

    df_tronque = pd.concat(frames, ignore_index=True) if frames else df_preds.iloc[0:0]
    return df_tronque, n_points_cible


def _recalculer_metriques_equite(df_preds_tronque, ordre_col):
    """Recomputes RMSE/MAE/n_points on the truncated predictions (fairness)."""
    if df_preds_tronque.empty:
        return pd.DataFrame()

    resultats = []
    for (source_id, source_name, leg_id, h, order), df_h in df_preds_tronque.groupby(
        ["SourceId", "SourceName", "Leg", "horizon_hours", ordre_col]
    ):
        rmse = np.sqrt(mean_squared_error(df_h["y_true"], df_h["y_pred"]))
        mae  = mean_absolute_error(df_h["y_true"], df_h["y_pred"])
        mean_y_h = np.abs(df_h["y_true"].mean())
        resultats.append({
            "SourceId": source_id,
            "SourceName": source_name,
            "Leg": leg_id,
            ordre_col: order,
            "horizon_hours": h,
            "MAE_horizon": mae,
            "RMSE_horizon": round(rmse, 2),
            "NMAE_horizon": mae / mean_y_h if mean_y_h != 0 else np.nan,
            "NRMSE_horizon": rmse / mean_y_h if mean_y_h != 0 else np.nan,
            "n_points": len(df_h),
        })
    return pd.DataFrame(resultats)

In [ ]:
def grid_search_arimax_order(
    df,
    p_values=[0, 1, 2],
    d_values=[0],
    q_values=[0, 1, 2],
    train_size=0.8,
    val_size=0.10,
    split_mode="train_val_test",   # "train_val_test" (80/10/10, used in the report) or "train_test" (former 90/10 protocol)
    timestamp_col="TimestampUTC",
    target_col="Value",            # consumption to predict (kW)
    exog_col="AIR_TEMPERATURE_2M",
    source_col="SourceId",
    source_name_col="SourceName",
    leg_col="Leg",
    freq="2H",
    criterion="RMSE_no_scaler",
    min_obs=30,
    target_agg="mean",
    verbose=True,
    versions_to_run=["V0", "V2"],
    h_steps_v2=1,
    v0_horizons_hours=None,
    v2_horizons_hours=None,
):
    """
    ARIMAX grid search, run independently for each SourceId of a one-leg dataframe.

    Two prediction modes are evaluated:
      - V0 (one-shot): the model is fitted on train, coefficients are frozen, and the
        whole evaluation period is forecast at once (no real value is fed back).
      - V2 (rolling):  same fitted model, but after each block of `h_steps_v2` steps the
        real observed values are appended to the model state (`.append(refit=False)`),
        without re-estimating the coefficients.

    The order (p, d, q) is selected on the VALIDATION set only; the test set is used
    once, for the final evaluation of the selected order.

    Parameters
    ----------
    df : DataFrame
        Data for one leg, containing several SourceId time series.
    p_values, d_values, q_values : list
        Candidate ARIMAX orders.
    train_size, val_size : float
        Chronological split proportions (test = remainder).
    split_mode : str
        "train_val_test" (order selected on validation) or "train_test"
        (former protocol: order selected directly on test, kept for comparison).
    freq : str
        Resampling frequency, e.g. "2H".
    criterion : str
        Metric used to sort the grid results ("RMSE_no_scaler" = V0 RMSE, "RMSE_rolling" = V2 RMSE, ...).
    min_obs : int
        Minimum number of observations required per SourceId.
    target_agg : str
        Aggregation for duplicated timestamps ("mean": the value is repeated after the
        merge with reefer positions).
    versions_to_run : list
        Subset of ["V0", "V2"].
    h_steps_v2 : int
        Block size (in time steps) of the V2 rolling update.
    v0_horizons_hours, v2_horizons_hours : list or None
        If given, multi-horizon evaluation (e.g. [2, 6, 12, 24, 48]) of the selected model.

    Returns
    -------
    df_best_results, df_all_grid_results, df_all_predictions (validation), best_models,
    df_horizon_results, df_horizon_predictions (V0 multi-horizon),
    df_v2_horizon_results, df_v2_horizon_predictions (V2 multi-horizon),
    df_all_predictions_test (test predictions V0 / V2),
    total_time_mh_v0, total_time_mh_v2, total_time_val_v0, total_time_val_v2,
    time_per_horizon_v0, time_per_horizon_v2 (execution times)
    """

    valid_criteria = [
        "MAE_no_scaler", "RMSE_no_scaler", "NMAE_no_scaler", "NRMSE_no_scaler",
        "MAE_rolling",   "RMSE_rolling",   "NMAE_rolling",   "NRMSE_rolling",
    ]
    if criterion not in valid_criteria:
        raise ValueError(f"criterion must be one of {valid_criteria}")

    df_work = df.copy()
    df_work[timestamp_col] = pd.to_datetime(df_work[timestamp_col])

    all_best_results = []        # one row per SourceId: best order and metrics
    all_grid_results = []        # one DataFrame per SourceId: all tested orders
    all_predictions = []         # validation predictions of the best models
    all_predictions_test = []    # test predictions of the best models
    best_models = {}             # {SourceId: {"no_scaler": V0 fit, "rolling": V2 fit}}
    all_horizon_results = []     # V0 multi-horizon metrics
    all_horizon_preds = []       # V0 multi-horizon predictions
    all_v2_horizon_results = []  # V2 multi-horizon metrics
    all_v2_horizon_preds = []    # V2 multi-horizon predictions
    freq_hours = int(''.join(filter(str.isdigit, freq))) if freq else 2  # "2H" -> 2

    total_time_mh_v0 = 0.0
    total_time_mh_v2 = 0.0
    total_time_val_v0 = 0.0
    total_time_val_v2 = 0.0
    time_per_horizon_v0 = {}     # {horizon_hours: cumulated time over all sources}
    time_per_horizon_v2 = {}

    orders = list(itertools.product(p_values, d_values, q_values))

    # Loop over each SourceId independently
    for source_id, group in df_work.groupby(source_col):

        try:
            if verbose:
                print(f"\n========== Processing SourceId: {source_id} ==========")

            group = group.copy()

            # Keep useful columns only
            useful_cols = [timestamp_col, target_col, exog_col]
            if source_name_col in group.columns:
                useful_cols.append(source_name_col)
            if leg_col in group.columns:
                useful_cols.append(leg_col)

            group = group[useful_cols].dropna(subset=[timestamp_col, target_col, exog_col])

            if len(group) < min_obs:
                if verbose:
                    print(f"Skipping SourceId {source_id}: not enough raw observations")
                continue

            # Metadata
            source_name = group[source_name_col].iloc[0] if source_name_col in group.columns else None
            leg_id = group[leg_col].iloc[0] if leg_col in group.columns else None

            # Aggregate duplicated timestamps (one row per timestamp per SourceId)
            agg_dict = {target_col: target_agg, exog_col: "mean"}
            group = (
                group
                .groupby(timestamp_col, as_index=False)
                .agg(agg_dict)
            )
            group = group.sort_values(timestamp_col)
            group = group.set_index(timestamp_col)

            # Resample to a regular frequency
            if freq is not None:
                group = group.resample(freq).agg(agg_dict)
                group = group.interpolate(method="time")
                group = group.dropna(subset=[target_col, exog_col])

            if len(group) < min_obs:
                if verbose:
                    print(f"Skipping SourceId {source_id}: not enough observations after resampling")
                continue

            y = group[target_col]
            X = group[[exog_col]]
            n = len(group)

            if split_mode == "train_val_test":
                # Current protocol: train / validation / test (e.g. 80/10/10).
                # The order (p,d,q) is selected on VALIDATION only; the TEST set is
                # never used for selection (no optimistic bias).
                train_end = int(n * train_size)
                val_end = int(n * (train_size + val_size))

                y_train, X_train = y.iloc[:train_end], X.iloc[:train_end]
                y_val, X_val = y.iloc[train_end:val_end], X.iloc[train_end:val_end]
                y_test, X_test = y.iloc[val_end:], X.iloc[val_end:]

            elif split_mode == "train_test":
                # Former protocol: train / test only (e.g. 90/10). The order was selected
                # directly on the test set (optimistic bias), which is why it was replaced
                # by the train/validation/test protocol above.
                train_end = int(n * train_size)
                val_end = train_end

                y_train, X_train = y.iloc[:train_end], X.iloc[:train_end]
                y_test, X_test = y.iloc[train_end:], X.iloc[train_end:]
                y_val, X_val = y_test, X_test

            else:
                raise ValueError("split_mode must be 'train_val_test' or 'train_test'")

            if len(y_train) < 5 or len(y_val) < 1 or len(y_test) < 1:
                if verbose:
                    print(f"Skipping SourceId {source_id}: invalid train/test split")
                continue

            source_grid_results = []

            # Independent best model for each version
            best_score_v0 = np.inf; best_order_v0 = None; best_y_pred_v0 = None; best_fit_v0 = None
            best_score_v2 = np.inf; best_order_v2 = None; best_y_pred_v2 = None
            best_v0_horizon_rows = []
            best_v2_horizon_rows = []
            best_prediction = None
            fitted_models_v0 = {}  # {order: fitted V0 model}
            fitted_models_v2 = {}  # {order: fitted V2 model}

            split_time = group.index[train_end]

            # Grid search for this SourceId (evaluated on validation)
            for order in orders:
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")

                        mean_y = np.abs(y_val.mean())

                        # ---------------- V0 — one-shot forecast ----------------
                        if "V0" in versions_to_run:
                            t0_val_v0 = time.time()
                            model_no_scaler = SARIMAX(y_train, exog=X_train, order=order,
                                                      enforce_stationarity=False, enforce_invertibility=False)
                            model_fit_no_scaler = model_no_scaler.fit(disp=False)
                            fitted_models_v0[order] = model_fit_no_scaler
                            y_pred_no_scaler = model_fit_no_scaler.forecast(steps=len(y_val), exog=X_val)
                            mae_no_scaler = mean_absolute_error(y_val, y_pred_no_scaler)
                            rmse_no_scaler = np.sqrt(mean_squared_error(y_val, y_pred_no_scaler))
                            nmae_no_scaler = mae_no_scaler / mean_y if mean_y != 0 else np.nan
                            nrmse_no_scaler = rmse_no_scaler / mean_y if mean_y != 0 else np.nan
                            total_time_val_v0 += time.time() - t0_val_v0
                        else:
                            model_fit_no_scaler = None
                            y_pred_no_scaler = pd.Series(np.full(len(y_val), np.nan), index=y_val.index)
                            mae_no_scaler = rmse_no_scaler = nmae_no_scaler = nrmse_no_scaler = np.nan

                        # ---------------- V2 — rolling block-h forecast ----------------
                        if "V2" in versions_to_run:
                            # 1. fit the ARIMAX model ONCE on y_train
                            # 2. forecast a block of h_steps_v2 steps
                            # 3. wait for the real values of that block
                            # 4. append them to the model state (.append(refit=False)),
                            #    without re-estimating the coefficients
                            # 5. move the origin forward by h_steps_v2 and repeat
                            t0_val_v2 = time.time()
                            n_val = len(y_val)

                            with warnings.catch_warnings():
                                warnings.simplefilter("ignore")
                                model_roll = SARIMAX(y_train.values, exog=X_train.values, order=order,
                                                     enforce_stationarity=False, enforce_invertibility=False)
                                model_fit_roll = model_roll.fit(disp=False)
                                fitted_models_v2[order] = model_fit_roll

                            y_pred_rolling_list = []          # predictions of all blocks
                            ts_rolling_list = []              # target timestamps
                            current_fit_roll = model_fit_roll  # model state, updated after each block

                            origin = 0
                            while origin < n_val:
                                # the last block may be shorter than h_steps_v2
                                end = min(origin + h_steps_v2, n_val)

                                exog_block = X_val.iloc[origin:end].values
                                with warnings.catch_warnings():
                                    warnings.simplefilter("ignore")
                                    forecast_block = current_fit_roll.forecast(steps=end - origin, exog=exog_block)

                                y_pred_rolling_list.extend(np.array(forecast_block).flatten().tolist())
                                ts_rolling_list.extend(y_val.index[origin:end].tolist())

                                # feed back the real observed values of this block
                                endog_block = y_val.iloc[origin:end].values
                                with warnings.catch_warnings():
                                    warnings.simplefilter("ignore")
                                    current_fit_roll = current_fit_roll.append(
                                        endog=endog_block, exog=exog_block, refit=False
                                    )

                                origin = end

                            if len(ts_rolling_list) > 0:
                                y_pred_rolling = pd.Series(y_pred_rolling_list, index=pd.DatetimeIndex(ts_rolling_list))
                                y_true_rolling = y_val.loc[y_pred_rolling.index]
                                mae_rolling = mean_absolute_error(y_true_rolling, y_pred_rolling)
                                rmse_rolling = np.sqrt(mean_squared_error(y_true_rolling, y_pred_rolling))
                                nmae_rolling = mae_rolling / mean_y if mean_y != 0 else np.nan
                                nrmse_rolling = rmse_rolling / mean_y if mean_y != 0 else np.nan
                            else:
                                y_pred_rolling = pd.Series(np.full(len(y_val), np.nan), index=y_val.index)
                                mae_rolling = rmse_rolling = nmae_rolling = nrmse_rolling = np.nan
                            total_time_val_v2 += time.time() - t0_val_v2
                        else:
                            y_pred_rolling = pd.Series(np.full(len(y_val), np.nan), index=y_val.index)
                            mae_rolling = rmse_rolling = nmae_rolling = nrmse_rolling = np.nan

                        row = {
                            "Leg": leg_id,
                            "SourceId": source_id,
                            "SourceName": source_name,
                            "order": order,
                            "p": order[0],
                            "d": order[1],
                            "q": order[2],
                            # V0 — one-shot
                            "MAE_no_scaler": mae_no_scaler,
                            "RMSE_no_scaler": rmse_no_scaler,
                            "NMAE_no_scaler": nmae_no_scaler,
                            "NRMSE_no_scaler": nrmse_no_scaler,
                            # V2 — rolling
                            "MAE_rolling": mae_rolling,
                            "RMSE_rolling": rmse_rolling,
                            "NMAE_rolling": nmae_rolling,
                            "NRMSE_rolling": nrmse_rolling,
                            "n_obs": len(group),
                            "n_train": len(y_train),
                            "n_val": len(y_val),
                            "n_test": len(y_test),
                        }
                        source_grid_results.append(row)

                        score = row[criterion]
                        if verbose:
                            print(f"SourceId {source_id} | order {order} | {criterion}: {score:.4f}")

                        # V0 — independent selection
                        if "V0" in versions_to_run and pd.notna(rmse_no_scaler) and rmse_no_scaler < best_score_v0:
                            best_score_v0 = rmse_no_scaler
                            best_order_v0 = order
                            best_fit_v0 = model_fit_no_scaler
                            best_y_pred_v0 = y_pred_no_scaler

                        # V2 — independent selection
                        if "V2" in versions_to_run and pd.notna(rmse_rolling) and rmse_rolling < best_score_v2:
                            best_score_v2 = rmse_rolling
                            best_order_v2 = order
                            best_y_pred_v2 = y_pred_rolling

                except Exception as e:
                    if verbose:
                        print(f"SourceId {source_id} | order {order} failed: {e}")

            # ============================================================
            # FINAL EVALUATION ON TEST — V0 and V2
            # (models already fitted on train, NO re-fitting)
            # ============================================================

            # ---------- V0 one-shot on test ----------
            y_pred_test_v0 = None
            rmse_test_v0 = np.nan
            mae_test_v0 = np.nan

            if best_order_v0 is not None and best_order_v0 in fitted_models_v0:
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        y_pred_test_v0 = fitted_models_v0[best_order_v0].forecast(
                            steps=len(y_test), exog=X_test
                        )
                    rmse_test_v0 = np.sqrt(mean_squared_error(y_test, y_pred_test_v0))
                    mae_test_v0 = mean_absolute_error(y_test, y_pred_test_v0)
                except Exception as e:
                    if verbose:
                        print(f"V0 test failed for SourceId {source_id}: {e}")

            # ---------- V2 rolling on test ----------
            y_pred_test_v2 = None
            rmse_test_v2 = np.nan
            mae_test_v2 = np.nan

            if "V2" in versions_to_run and best_order_v2 is not None and best_order_v2 in fitted_models_v2:
                try:
                    n_test = len(y_test)
                    current_fit_test = fitted_models_v2[best_order_v2]
                    y_pred_list_test = []
                    ts_list_test = []
                    origin = 0

                    while origin < n_test:
                        end = min(origin + h_steps_v2, n_test)
                        exog_block = X_test.iloc[origin:end].values

                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            forecast_block = current_fit_test.forecast(steps=end - origin, exog=exog_block)

                        y_pred_list_test.extend(np.array(forecast_block).flatten().tolist())
                        ts_list_test.extend(y_test.index[origin:end].tolist())

                        endog_block = y_test.iloc[origin:end].values
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            current_fit_test = current_fit_test.append(
                                endog=endog_block, exog=exog_block, refit=False
                            )
                        origin = end

                    y_pred_test_v2 = pd.Series(y_pred_list_test, index=pd.DatetimeIndex(ts_list_test))
                    y_true_test_v2 = y_test.loc[y_pred_test_v2.index]
                    rmse_test_v2 = np.sqrt(mean_squared_error(y_true_test_v2, y_pred_test_v2))
                    mae_test_v2 = mean_absolute_error(y_true_test_v2, y_pred_test_v2)
                except Exception as e:
                    if verbose:
                        print(f"V2 test failed for SourceId {source_id}: {e}")

            if verbose:
                print(f"TEST | SourceId {source_id} | V0 RMSE={rmse_test_v0:.4f} | V2 RMSE={rmse_test_v2:.4f}")

            # ---------- Test predictions DataFrame ----------
            best_prediction_test = pd.DataFrame({
                timestamp_col: y_test.index,
                "y_true": y_test.values,
                "y_pred_v0": y_pred_test_v0 if y_pred_test_v0 is not None else np.full(len(y_test), np.nan),
            })
            # V2: aligned on timestamps
            if y_pred_test_v2 is not None:
                best_prediction_test["y_pred_v2"] = best_prediction_test[timestamp_col].map(y_pred_test_v2.to_dict())
            else:
                best_prediction_test["y_pred_v2"] = np.nan

            best_prediction_test["RMSE_test_V0"] = rmse_test_v0
            best_prediction_test["RMSE_test_V2"] = rmse_test_v2
            best_prediction_test["val_test_split"] = group.index[val_end]
            best_prediction_test["Leg"] = leg_id
            best_prediction_test["SourceId"] = source_id
            best_prediction_test["SourceName"] = source_name
            best_prediction_test["best_order_V0"] = str(best_order_v0)
            best_prediction_test["best_order_V2"] = str(best_order_v2)

            all_predictions_test.append(best_prediction_test)

            # ---------- Validation predictions of the best models ----------
            if best_y_pred_v0 is not None or best_y_pred_v2 is not None:
                best_prediction = pd.DataFrame({
                    timestamp_col: y_val.index,
                    "y_true": y_val.values,
                    "y_pred_no_scaler": best_y_pred_v0 if best_y_pred_v0 is not None else np.full(len(y_val), np.nan),
                    "y_pred_rolling": best_y_pred_v2 if best_y_pred_v2 is not None else np.full(len(y_val), np.nan),
                })
                best_prediction["split_time"] = split_time
                best_prediction["residual_no_scaler"] = best_prediction["y_true"] - best_prediction["y_pred_no_scaler"]
                best_prediction["residual_rolling"] = best_prediction["y_true"] - best_prediction["y_pred_rolling"]
                best_prediction["Leg"] = leg_id
                best_prediction["SourceId"] = source_id
                best_prediction["SourceName"] = source_name
                best_prediction["best_order_V0"] = str(best_order_v0)
                best_prediction["best_order_V2"] = str(best_order_v2)

            # ---------- V0 multi-horizon (sliding origins, best order, no re-fitting) ----------
            if v0_horizons_hours is not None and best_order_v0 is not None and best_order_v0 in fitted_models_v0:
                t0_mh_v0 = time.time()
                model_fit_final_v0 = fitted_models_v0[best_order_v0]
                n_test_local = len(y_test)
                best_v0_horizon_rows = []

                for h_hours in sorted(set(v0_horizons_hours)):
                    h_steps = h_hours // freq_hours
                    if h_steps <= 0:
                        continue

                    t0_h_v0 = time.time()
                    mean_y_h = np.abs(y_test.mean())
                    preds_h, trues_h, ts_h = [], [], []

                    for origin_idx in range(0, n_test_local - h_steps):
                        target_idx = origin_idx + h_steps
                        if target_idx >= n_test_local:
                            break
                        try:
                            X_block = X_test.iloc[origin_idx:origin_idx + h_steps].values
                            with warnings.catch_warnings():
                                warnings.simplefilter("ignore")
                                forecast_block = np.array(
                                    model_fit_final_v0.forecast(steps=h_steps, exog=X_block)
                                ).flatten()
                            # keep only the last point = exactly h hours ahead
                            preds_h.append(forecast_block[-1])
                            trues_h.append(y_test.iloc[target_idx])
                            ts_h.append(y_test.index[target_idx])
                        except Exception:
                            pass

                    time_per_horizon_v0[h_hours] = time_per_horizon_v0.get(h_hours, 0.0) + (time.time() - t0_h_v0)

                    if len(preds_h) == 0:
                        continue

                    rmse_h = np.sqrt(mean_squared_error(trues_h, preds_h))
                    mae_h = mean_absolute_error(trues_h, preds_h)

                    if verbose:
                        print(f"   V0 | SourceId {source_id} | h={h_hours}h | "
                              f"order={best_order_v0} | RMSE={rmse_h:.4f} | n_points={len(preds_h)}")

                    for k in range(len(preds_h)):
                        best_v0_horizon_rows.append({
                            "horizon_hours": h_hours,
                            "best_order_V0_horizon": str(best_order_v0),
                            "TimestampUTC": ts_h[k],
                            "y_true": trues_h[k],
                            "y_pred": preds_h[k],
                            "MAE_horizon": mae_h,
                            "RMSE_horizon": rmse_h,
                            "NMAE_horizon": mae_h / mean_y_h if mean_y_h != 0 else np.nan,
                            "NRMSE_horizon": rmse_h / mean_y_h if mean_y_h != 0 else np.nan,
                            "n_points": len(preds_h),
                        })

                total_time_mh_v0 += time.time() - t0_mh_v0

            # ---------- V2 multi-horizon (sliding origins, best order, no re-fitting) ----------
            if v2_horizons_hours is not None and best_order_v2 is not None and best_order_v2 in fitted_models_v2:
                t0_mh_v2 = time.time()
                model_fit_final_v2 = fitted_models_v2[best_order_v2]
                n_test_local = len(y_test)
                best_v2_horizon_rows = []

                for h_hours in sorted(set(v2_horizons_hours)):
                    h_steps = h_hours // freq_hours
                    if h_steps <= 0:
                        continue

                    t0_h_v2 = time.time()
                    mean_y_h = np.abs(y_test.mean())
                    preds_h, trues_h, ts_h = [], [], []
                    current_fit_h = model_fit_final_v2  # model state moving forward with the real values

                    for origin_idx in range(0, n_test_local - h_steps):
                        target_idx = origin_idx + h_steps
                        if target_idx >= n_test_local:
                            break
                        try:
                            # rolling blocks of h_steps_v2 steps until h_steps is reached
                            current_fit_origin = current_fit_h
                            steps_done = 0
                            last_pred = None
                            while steps_done < h_steps:
                                block_size = min(h_steps_v2, h_steps - steps_done)
                                exog_block = X_test.iloc[origin_idx + steps_done:origin_idx + steps_done + block_size].values
                                endog_block = y_test.iloc[origin_idx + steps_done:origin_idx + steps_done + block_size].values
                                with warnings.catch_warnings():
                                    warnings.simplefilter("ignore")
                                    forecast_block = np.array(
                                        current_fit_origin.forecast(steps=block_size, exog=exog_block)
                                    ).flatten()
                                last_pred = forecast_block[-1]
                                with warnings.catch_warnings():
                                    warnings.simplefilter("ignore")
                                    current_fit_origin = current_fit_origin.append(
                                        endog=endog_block, exog=exog_block, refit=False
                                    )
                                steps_done += block_size

                            # keep only the last point = exactly h hours ahead
                            preds_h.append(last_pred)
                            trues_h.append(y_test.iloc[target_idx])
                            ts_h.append(y_test.index[target_idx])
                        except Exception:
                            pass

                        # move the main model state forward by one step
                        try:
                            endog_origin = [y_test.iloc[origin_idx]]
                            exog_origin = X_test.iloc[[origin_idx]].values
                            with warnings.catch_warnings():
                                warnings.simplefilter("ignore")
                                current_fit_h = current_fit_h.append(
                                    endog=endog_origin, exog=exog_origin, refit=False
                                )
                        except Exception:
                            pass

                    time_per_horizon_v2[h_hours] = time_per_horizon_v2.get(h_hours, 0.0) + (time.time() - t0_h_v2)

                    if len(preds_h) == 0:
                        continue

                    rmse_h = np.sqrt(mean_squared_error(trues_h, preds_h))
                    mae_h = mean_absolute_error(trues_h, preds_h)

                    if verbose:
                        print(f"   V2 | SourceId {source_id} | h={h_hours}h | "
                              f"order={best_order_v2} | RMSE={rmse_h:.4f} | n_points={len(preds_h)}")

                    for k in range(len(preds_h)):
                        best_v2_horizon_rows.append({
                            "horizon_hours": h_hours,
                            "best_order_V2_horizon": str(best_order_v2),
                            "TimestampUTC": ts_h[k],
                            "y_true": trues_h[k],
                            "y_pred": preds_h[k],
                            "MAE_horizon": mae_h,
                            "RMSE_horizon": rmse_h,
                            "NMAE_horizon": mae_h / mean_y_h if mean_y_h != 0 else np.nan,
                            "NRMSE_horizon": rmse_h / mean_y_h if mean_y_h != 0 else np.nan,
                            "n_points": len(preds_h),
                        })

                total_time_mh_v2 += time.time() - t0_mh_v2

            if len(source_grid_results) == 0:
                if verbose:
                    print(f"No valid model found for SourceId {source_id}")
                continue

            df_source_grid = pd.DataFrame(source_grid_results)
            df_source_grid = df_source_grid.sort_values(criterion).reset_index(drop=True)

            best_row = df_source_grid.iloc[0].copy()
            best_row["best_order_V0"] = str(best_order_v0)
            best_row["best_order_V2"] = str(best_order_v2)
            best_row["best_score_V0"] = best_score_v0
            best_row["best_score_V2"] = best_score_v2
            best_row["selection_criterion"] = criterion

            all_best_results.append(best_row)
            all_grid_results.append(df_source_grid)

            if best_prediction is not None:
                all_predictions.append(best_prediction)

            # Store V0 multi-horizon results
            if v0_horizons_hours is not None and best_v0_horizon_rows:
                df_v0_h = pd.DataFrame(best_v0_horizon_rows)
                df_v0_h["SourceId"] = source_id
                df_v0_h["SourceName"] = source_name
                df_v0_h["Leg"] = leg_id
                all_horizon_results.append(df_v0_h.drop_duplicates(subset=["horizon_hours"])[[
                    "SourceId", "SourceName", "Leg", "best_order_V0_horizon",
                    "horizon_hours", "MAE_horizon", "RMSE_horizon", "NMAE_horizon", "NRMSE_horizon", "n_points",
                ]])
                all_horizon_preds.append(df_v0_h[[
                    "SourceId", "SourceName", "Leg", "best_order_V0_horizon",
                    "horizon_hours", "TimestampUTC", "y_true", "y_pred",
                ]])

            # Store V2 multi-horizon results
            if v2_horizons_hours is not None and best_v2_horizon_rows:
                df_v2_h = pd.DataFrame(best_v2_horizon_rows)
                df_v2_h["SourceId"] = source_id
                df_v2_h["SourceName"] = source_name
                df_v2_h["Leg"] = leg_id
                all_v2_horizon_results.append(df_v2_h.drop_duplicates(subset=["horizon_hours"])[[
                    "SourceId", "SourceName", "Leg", "best_order_V2_horizon",
                    "horizon_hours", "MAE_horizon", "RMSE_horizon", "NMAE_horizon", "NRMSE_horizon", "n_points",
                ]])
                all_v2_horizon_preds.append(df_v2_h[[
                    "SourceId", "SourceName", "Leg", "best_order_V2_horizon",
                    "horizon_hours", "TimestampUTC", "y_true", "y_pred",
                ]])

            best_models[source_id] = {
                "no_scaler": best_fit_v0,
                "rolling": fitted_models_v2.get(best_order_v2),
            }

            if verbose:
                print(f"Best SourceId {source_id}: V0 order={best_order_v0} RMSE={best_score_v0:.4f} | "
                      f"V2 order={best_order_v2} RMSE={best_score_v2:.4f}")

        except Exception as e:
            print(f"Error while processing SourceId {source_id}: {e}")

    # Final outputs
    def _concat(frames):
        return pd.concat(frames, ignore_index=True) if len(frames) > 0 else pd.DataFrame()

    df_best_results = pd.DataFrame(all_best_results) if len(all_best_results) > 0 else pd.DataFrame()
    df_all_grid_results = _concat(all_grid_results)
    df_all_predictions = _concat(all_predictions)
    df_all_predictions_test = _concat(all_predictions_test)
    df_horizon_results = _concat(all_horizon_results)
    df_horizon_predictions = _concat(all_horizon_preds)
    df_v2_horizon_results = _concat(all_v2_horizon_results)
    df_v2_horizon_predictions = _concat(all_v2_horizon_preds)

    # Fairness between horizons: every horizon is evaluated on the same number of points
    if v0_horizons_hours is not None and not df_horizon_predictions.empty:
        df_horizon_predictions, _ = _tronquer_predictions_equite(df_horizon_predictions, v0_horizons_hours)
        df_horizon_results = _recalculer_metriques_equite(df_horizon_predictions, "best_order_V0_horizon")

    if v2_horizons_hours is not None and not df_v2_horizon_predictions.empty:
        df_v2_horizon_predictions, _ = _tronquer_predictions_equite(df_v2_horizon_predictions, v2_horizons_hours)
        df_v2_horizon_results = _recalculer_metriques_equite(df_v2_horizon_predictions, "best_order_V2_horizon")

    return (df_best_results, df_all_grid_results, df_all_predictions, best_models,
            df_horizon_results, df_horizon_predictions, df_v2_horizon_results, df_v2_horizon_predictions,
            df_all_predictions_test, total_time_mh_v0, total_time_mh_v2, total_time_val_v0, total_time_val_v2,
            time_per_horizon_v0, time_per_horizon_v2)

In [ ]:
def plot_predictions(
    df_predictions_test,        # = df_all_predictions_test returned by grid_search_arimax_order
    df_full=None,               # full series (train + val + test) for historical context
    df_v0_horizon_preds=None,   # V0 multi-horizon predictions
    df_v2_horizon_preds=None,   # V2 multi-horizon predictions
    horizon_hours=None,         # if set, only this horizon is displayed
    split_mode="train_val_test",
    timestamp_col="TimestampUTC",
    source_col="SourceId",
    source_name_col="SourceName",
    leg_col="Leg",
    y_true_col="y_true",
    y_pred_v0_col="y_pred_v0",
    y_pred_v2_col="y_pred_v2",
    split_col="val_test_split",
    target_col="Value",
    freq="2H",
    target_agg="mean",
):
    """
    One figure per source: real consumption (history + test), train/test or
    validation/test split line, test predictions V0 one-shot (red) and V2 rolling
    (steel blue), and optional V0/V2 multi-horizon predictions.

    A curve (and its legend entry) is only added if the column exists and
    contains at least one non-NaN value.
    """

    if split_mode not in ("train_val_test", "train_test"):
        raise ValueError("split_mode must be 'train_val_test' or 'train_test'")

    horizon_colors = {2: "red", 6: "orange", 12: "green", 24: "purple", 48: "pink", 60: "yellow"}

    split_annotation = "Validation/Test split" if split_mode == "train_val_test" else "Train/Test split"

    df_plot_all = df_predictions_test.copy()
    df_plot_all[timestamp_col] = pd.to_datetime(df_plot_all[timestamp_col])
    if split_col in df_plot_all.columns:
        df_plot_all[split_col] = pd.to_datetime(df_plot_all[split_col])

    if df_full is not None:
        df_full_plot = df_full.copy()
        df_full_plot[timestamp_col] = pd.to_datetime(df_full_plot[timestamp_col])
    else:
        df_full_plot = None

    df_v0_h_filtered = df_v0_horizon_preds
    df_v2_h_filtered = df_v2_horizon_preds
    if horizon_hours is not None:
        if df_v0_horizon_preds is not None:
            df_v0_h_filtered = df_v0_horizon_preds[df_v0_horizon_preds["horizon_hours"] == horizon_hours]
        if df_v2_horizon_preds is not None:
            df_v2_h_filtered = df_v2_horizon_preds[df_v2_horizon_preds["horizon_hours"] == horizon_hours]

    def has_data(df, col):
        """True if the column exists and contains at least one non-NaN value."""
        return col in df.columns and df[col].notna().any()

    for source_id, group in df_plot_all.groupby(source_col):

        group = group.sort_values(timestamp_col)
        source_name = group[source_name_col].iloc[0] if source_name_col in group.columns else ""
        leg_id = group[leg_col].iloc[0] if leg_col in group.columns else ""
        current_split_time = (
            pd.to_datetime(group[split_col].iloc[0]).to_pydatetime() if split_col in group.columns else None
        )

        title = f"ARIMAX prediction - SourceId {source_id}"
        if source_name != "":
            title += f" | {source_name}"
        if leg_id != "":
            title += f" | Leg {leg_id}"
        if "best_order_V0" in group.columns:
            title += f" | V0 order={group['best_order_V0'].iloc[0]}"
        if has_data(group, y_pred_v2_col) and "best_order_V2" in group.columns:
            title += f" | V2 order={group['best_order_V2'].iloc[0]}"
        if horizon_hours is not None:
            title += f" | horizon={int(horizon_hours)}h"

        fig = go.Figure()

        # Real consumption — one continuous curve (history + test)
        if df_full_plot is not None:
            full_source = df_full_plot[df_full_plot[source_col].astype(str) == str(source_id)].copy()
            if leg_col in full_source.columns and leg_id != "":
                full_source = full_source[full_source[leg_col].astype(str) == str(leg_id)]
            if not full_source.empty:
                full_source = (
                    full_source.groupby(timestamp_col, as_index=False)
                    .agg({target_col: target_agg})
                    .sort_values(timestamp_col)
                    .set_index(timestamp_col)
                )
                if freq is not None:
                    full_source = (
                        full_source.resample(freq).agg({target_col: target_agg})
                        .interpolate(method="time").dropna(subset=[target_col])
                    )
                full_source = full_source.reset_index()
                fig.add_trace(go.Scatter(
                    x=full_source[timestamp_col], y=full_source[target_col],
                    mode="lines", name="Real consumption", line=dict(color="deepskyblue"),
                ))
        else:
            fig.add_trace(go.Scatter(
                x=group[timestamp_col], y=group[y_true_col],
                mode="lines", name="Real consumption (test)", line=dict(color="deepskyblue"),
            ))

        # V0 — one-shot test prediction
        if has_data(group, y_pred_v0_col):
            fig.add_trace(go.Scatter(
                x=group[timestamp_col], y=group[y_pred_v0_col],
                mode="lines", name="V0 (one-shot)", line=dict(color="red", width=1.5),
            ))

        # V2 — rolling test prediction
        if has_data(group, y_pred_v2_col):
            fig.add_trace(go.Scatter(
                x=group[timestamp_col], y=group[y_pred_v2_col],
                mode="lines", name="V2 (rolling)", line=dict(color="steelblue"), connectgaps=False,
            ))

        # Multi-horizon predictions
        for df_h, version, dash, symbol, fixed_color in (
            (df_v0_h_filtered, "V0", "dash", "circle", "red"),
            (df_v2_h_filtered, "V2", "dot", "diamond", "steelblue"),
        ):
            if df_h is None:
                continue
            horizon_source = df_h[df_h[source_col] == source_id].copy()
            if horizon_source.empty:
                continue
            horizon_source[timestamp_col] = pd.to_datetime(horizon_source[timestamp_col])
            for h_hours, grp_h in horizon_source.groupby("horizon_hours"):
                grp_h = grp_h.dropna(subset=["y_pred"]).sort_values(timestamp_col)
                if grp_h.empty:
                    continue
                color = fixed_color if horizon_hours is not None else horizon_colors.get(int(h_hours), "gray")
                fig.add_trace(go.Scatter(
                    x=grp_h[timestamp_col], y=grp_h["y_pred"],
                    mode="lines+markers", name=f"{version} multi-horizon h={int(h_hours)}h",
                    line=dict(color=color, dash=dash), marker=dict(size=8, symbol=symbol),
                ))

        # Split line
        if current_split_time is not None:
            fig.add_shape(type="line", x0=current_split_time, x1=current_split_time,
                          y0=0, y1=1, xref="x", yref="paper")
            fig.add_annotation(x=current_split_time, y=1, xref="x", yref="paper",
                               text=split_annotation, showarrow=False, yanchor="bottom")

        fig.update_layout(title=title, xaxis_title="Time", yaxis_title="Consumption (kW)",
                          template="plotly_white", hovermode="x unified", width=1100, height=500)
        fig.show()

### 4.2 One-shot prediction (V0)

Coefficients are frozen after training and the whole test period is predicted at once, without feeding back any real observation.

In [ ]:
# ============================================================
# ARIMAX — V0 ONLY (Leg 30)
# ============================================================

leg_num = 30
df_leg_current = df_final_all[df_final_all["Leg"] == leg_num]

(_, _, _, _, _, _, _, _,
 df_predictions_test_v0, _, _, _, _, _, _) = grid_search_arimax_order(
    df_leg_current,
    p_values=[0, 1, 2], d_values=[0], q_values=[0, 1, 2],
    freq="2h", criterion="RMSE_no_scaler",
    min_obs=30, target_agg="mean", verbose=False,
    versions_to_run=["V0"],
    split_mode="train_val_test", train_size=0.8, val_size=0.10,
    h_steps_v2=1,
)

# ---------------- RMSE TABLE — all sources, V0 ----------------
df_rmse_v0 = df_predictions_test_v0.drop_duplicates(subset=["SourceId"])[
    ["SourceId", "SourceName", "best_order_V0", "RMSE_test_V0"]
].reset_index(drop=True)

# --- mean real consumption over the test period, per source ---
df_conso_moy_test = (
    df_predictions_test_v0.groupby("SourceId")["y_true"]
    .mean()
    .rename("conso_moyenne_test_kW")
    .reset_index()
)

df_rmse_v0 = df_rmse_v0.merge(df_conso_moy_test, on="SourceId", how="left")

# --- relative RMSE, in % of the mean test consumption ---
df_rmse_v0["RMSE_relatif_%"] = (
    df_rmse_v0["RMSE_test_V0"] / df_rmse_v0["conso_moyenne_test_kW"] * 100
)

df_rmse_v0["RMSE_test_V0"] = df_rmse_v0["RMSE_test_V0"].round(2)
df_rmse_v0["conso_moyenne_test_kW"] = df_rmse_v0["conso_moyenne_test_kW"].round(2)
df_rmse_v0["RMSE_relatif_%"] = df_rmse_v0["RMSE_relatif_%"].round(1)

df_rmse_v0 = df_rmse_v0.rename(columns={"RMSE_test_V0": "RMSE_V0"})

print(f"\n=== RMSE TABLE — ARIMAX V0 — Leg {leg_num} ===")
print(df_rmse_v0.to_string(index=False))
print(f"\nMean RMSE V0 (all sources) : {df_rmse_v0['RMSE_V0'].mean():.2f}")
print(f"Mean relative RMSE (all sources) : {df_rmse_v0['RMSE_relatif_%'].mean():.1f}%")

# ---------------- FIGURES — real consumption vs V0 ----------------
plot_predictions(
    df_predictions_test=df_predictions_test_v0,
    df_full=df_leg_current,
    split_mode="train_val_test",
    freq="2h",
    target_agg="mean",
    y_pred_v0_col="y_pred_v0",
)


=== RMSE TABLE — ARIMAX V0 — Leg 30 ===
 SourceId                                             SourceName best_order_V0  RMSE_V0  conso_moyenne_test_kW  RMSE_relatif_%
      103  RSB1_Switchboard.RSB1-S2_Reefer_RP7_-_Bay_22_Deck_Std     (2, 0, 0)     3.43                  17.89            19.2
      116  RSB2_Switchboard.RSB2-S4_Reefer_RP10_-_Bay_22_Deck_PS     (2, 0, 2)     4.30                  11.30            38.1
      126 RSB3_Switchboard.RSB3-S4_Reefer_RP21_-_Bay_46_deck_std     (1, 0, 1)     4.24                  12.79            33.2
      127 RSB3_Switchboard.RSB3-S4_Reefer_RP25_-_Bay_50_deck_Std     (2, 0, 1)     3.99                  50.00             8.0
      128 RSB3_Switchboard.RSB3-S4_Reefer_RP29_-_Bay_54_Deck_Std     (1, 0, 1)     2.85                  86.74             3.3
      129  RSB4_Switchboard.RSB4-S1_Reefer_RP16_-_Bay_38_Deck_PS     (2, 0, 1)     5.58                  63.13             8.8
      131  RSB4_Switchboard.RSB4-S2_Reefer_RP22_-_Bay_46_Deck_PS     (

### 4.3 Impact of the prediction horizon (V0)

Same V0 trajectory, evaluated on the first 2 h, 6 h, 12 h, 24 h and 48 h of the test period.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

HORIZONS_H = [2, 6, 12, 24, 48]
FREQ_HOURS = 2
leg_num = 30
df_leg_current = df_final_all[df_final_all["Leg"] == leg_num]

# ============================================================
# 0. V0 multi-horizon + V0 one-shot (for the mean test consumption)
# ============================================================

(_, _, _, _,
 df_horizon_V0_metrics, df_horizon_V0_preds,
 _, _,
 df_predictions_test_v0, _, _, _, _, _, _) = grid_search_arimax_order(
    df_leg_current,
    p_values=[0, 1, 2], d_values=[0], q_values=[0, 1, 2],
    freq="2h", criterion="RMSE_no_scaler",
    min_obs=30, target_agg="mean", verbose=False,
    versions_to_run=["V0"],
    split_mode="train_val_test", train_size=0.8, val_size=0.10,
    v0_horizons_hours=HORIZONS_H,
)

# conso_moyenne_test: mean of y_true over the WHOLE test period (V0 one-shot), per source
df_conso_moyenne = (
    df_predictions_test_v0.groupby("SourceId")["y_true"]
    .mean()
    .rename("conso_moyenne_test")
    .reset_index()
)


# ============================================================
# 1. RMSE + relative RMSE per source and per horizon
# ============================================================

def calculer_rmse_par_horizon(df_horizon_V0_preds, df_conso_moyenne):
    rows = []
    for (source_id, horizon), grp in df_horizon_V0_preds.groupby(["SourceId", "horizon_hours"]):
        grp = grp.dropna(subset=["y_true", "y_pred"])
        if grp.empty:
            continue
        rmse = np.sqrt(np.mean((grp["y_true"] - grp["y_pred"]) ** 2))
        rows.append({"SourceId": source_id, "horizon_hours": horizon, "RMSE": rmse})

    df_rmse = pd.DataFrame(rows).merge(df_conso_moyenne, on="SourceId", how="left")
    df_rmse["RMSE_relatif_pct"] = df_rmse["RMSE"] / df_rmse["conso_moyenne_test"] * 100
    df_rmse["RMSE"] = df_rmse["RMSE"].round(2)
    df_rmse["RMSE_relatif_pct"] = df_rmse["RMSE_relatif_pct"].round(1)
    return df_rmse[["SourceId", "horizon_hours", "RMSE", "RMSE_relatif_pct"]]


def pivoter_tableau_rmse(df_rmse):
    df_rmse = df_rmse.copy()
    df_rmse["cellule"] = df_rmse["RMSE"].astype(str) + " / " + df_rmse["RMSE_relatif_pct"].astype(str)

    df_pivot = df_rmse.pivot(index="SourceId", columns="horizon_hours", values="cellule")
    df_pivot = df_pivot[[h for h in HORIZONS_H if h in df_pivot.columns]]
    df_pivot.columns = [f"{h}h" for h in df_pivot.columns]
    return df_pivot.sort_index().reset_index()


df_rmse_horizon = calculer_rmse_par_horizon(df_horizon_V0_preds, df_conso_moyenne)
df_tableau_pivot = pivoter_tableau_rmse(df_rmse_horizon)

print("=== RMSE / relative RMSE (%) per source and per horizon — ARIMAX V0 ===")
print(df_tableau_pivot.to_string(index=False))

=== RMSE / relative RMSE (%) per source and per horizon — ARIMAX V0 ===
 SourceId          2h          6h         12h         24h         48h
      103  1.73 / 9.7  2.6 / 14.5 2.78 / 15.5 3.92 / 21.9 3.34 / 18.7
      116 3.83 / 33.9 3.86 / 34.2 3.44 / 30.4 4.41 / 39.1 4.94 / 43.8
      126 3.49 / 27.3 4.05 / 31.6 3.89 / 30.4 5.55 / 43.4  1.04 / 8.1
      127   2.7 / 5.4  3.23 / 6.5  3.46 / 6.9   4.2 / 8.4  4.57 / 9.1
      128  2.76 / 3.2   3.0 / 3.5  2.73 / 3.2  2.17 / 2.5  3.73 / 4.3
      129  4.97 / 7.9  4.12 / 6.5  3.55 / 5.6   4.6 / 7.3 6.67 / 10.6
      131  2.99 / 7.1  2.06 / 4.9  2.08 / 5.0   3.0 / 7.2  3.46 / 8.3
      132 1.04 / 12.5 1.02 / 12.2 1.66 / 19.9  0.82 / 9.8 2.05 / 24.7
      138  4.03 / 4.5  2.79 / 3.1  2.88 / 3.2  6.64 / 7.4  5.76 / 6.4
      142 0.92 / 14.8 0.98 / 15.8 1.03 / 16.6 1.74 / 28.1 1.69 / 27.2
      146 2.89 / 14.7 3.46 / 17.6 2.69 / 13.7 4.96 / 25.3 4.54 / 23.2
      150  0.72 / 6.1  0.72 / 6.2  0.43 / 3.7  0.77 / 6.5 1.46 / 12.4


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

def plot_predictions_with_horizon_arrows(
    df_predictions_test,
    df_full,
    horizons_h=(2, 6, 12, 24, 48),
    timestamp_col="TimestampUTC",
    source_col="SourceId",
    source_name_col="SourceName",
    leg_col="Leg",
    y_true_col="y_true",
    y_pred_v0_col="y_pred_v0",
    split_col="val_test_split",
    target_col="Value",
    freq="2h",
    target_agg="mean",
    horizon_colors=None,
):
    """
    Real consumption + V0 prediction (solid line), with horizontal double arrows
    below the curve for each horizon (2h/6h/12h/24h/48h), from the start of the
    test period (val_test_split) to test_start + h.
    """
    if horizon_colors is None:
        horizon_colors = {2: "red", 6: "green", 12: "blue", 24: "purple", 48: "brown"}

    df_plot_all = df_predictions_test.copy()
    df_plot_all[timestamp_col] = pd.to_datetime(df_plot_all[timestamp_col])
    if split_col in df_plot_all.columns:
        df_plot_all[split_col] = pd.to_datetime(df_plot_all[split_col])

    df_full_plot = df_full.copy()
    df_full_plot[timestamp_col] = pd.to_datetime(df_full_plot[timestamp_col])

    for source_id, group in df_plot_all.groupby(source_col):
        group = group.sort_values(timestamp_col)

        source_name = group[source_name_col].iloc[0] if source_name_col in group.columns else ""
        leg_id = group[leg_col].iloc[0] if leg_col in group.columns else ""

        test_start = (
            pd.to_datetime(group[split_col].iloc[0])
            if split_col in group.columns else group[timestamp_col].iloc[0]
        )

        title = f"ARIMAX prediction — SourceId {source_id}"
        if source_name != "":
            title += f" | {source_name}"
        if leg_id != "":
            title += f" | Leg {leg_id}"

        fig = go.Figure()

        # ---- Real consumption (history + test, single curve) ----
        full_source = df_full_plot[df_full_plot[source_col].astype(str) == str(source_id)].copy()
        if leg_col in full_source.columns and leg_id != "":
            full_source = full_source[full_source[leg_col].astype(str) == str(leg_id)]

        if not full_source.empty:
            full_source = (
                full_source.groupby(timestamp_col, as_index=False)
                .agg({target_col: target_agg})
                .sort_values(timestamp_col)
                .set_index(timestamp_col)
            )
            if freq is not None:
                full_source = (
                    full_source.resample(freq).agg({target_col: target_agg})
                    .interpolate(method="time").dropna(subset=[target_col])
                )
            full_source = full_source.reset_index()

            fig.add_trace(go.Scatter(
                x=full_source[timestamp_col], y=full_source[target_col],
                mode="lines", name="real consumption",
                line=dict(color="black")
            ))

        # ---- V0 predicted consumption (solid line, orange) ----
        if y_pred_v0_col in group.columns and group[y_pred_v0_col].notna().any():
            fig.add_trace(go.Scatter(
                x=group[timestamp_col], y=group[y_pred_v0_col],
                mode="lines", name="predicted consumption",
                line=dict(color="orange", width=2)
            ))

        # ---- "test start" line ----
        fig.add_shape(
            type="line", x0=test_start, x1=test_start, y0=0, y1=1,
            xref="x", yref="paper", line=dict(color="black", width=1)
        )
        fig.add_annotation(
            x=test_start, y=1, xref="x", yref="paper",
            text="test start", showarrow=False, yanchor="bottom"
        )

        # ---- Double arrows stacked below the curve, one per horizon ----
        y_min_candidates = [group[y_true_col].min()]
        if not full_source.empty:
            y_min_candidates.append(full_source[target_col].min())
        if y_pred_v0_col in group.columns:
            y_min_candidates.append(group[y_pred_v0_col].min())
        y_min = np.nanmin(y_min_candidates)

        y_range = (
            full_source[target_col].max() - full_source[target_col].min()
            if not full_source.empty else group[y_true_col].max() - group[y_true_col].min()
        )
        y_step = 0.08 * y_range

        for i, h in enumerate(horizons_h):
            t_h = test_start + pd.Timedelta(hours=h)
            color = horizon_colors.get(h, "gray")
            y_arrow = y_min - (i + 1) * y_step
            t_mid = test_start + pd.Timedelta(hours=h / 2)

            # coloured vertical line at the horizon
            fig.add_shape(
                type="line", x0=t_h, x1=t_h, y0=0, y1=1,
                xref="x", yref="paper", line=dict(color=color, width=1)
            )

            # double arrow = two annotations from the middle towards each end
            for x_end in (test_start, t_h):
                fig.add_annotation(
                    x=x_end, y=y_arrow, ax=t_mid, ay=y_arrow,
                    xref="x", yref="y", axref="x", ayref="y",
                    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5,
                    arrowcolor=color, text=""
                )

            # "Xh" label above the arrow
            fig.add_annotation(
                x=t_mid, y=y_arrow, text=f"{h}h",
                showarrow=False, yshift=12,
                font=dict(color=color, size=12)
            )

        fig.update_layout(
            title=title,
            xaxis_title="Time",
            yaxis_title="Consumption",
            template="plotly_white",
            hovermode="x unified",
            width=1100,
            height=600,
        )
        fig.show()


# ---- Utilisation ----
plot_predictions_with_horizon_arrows(
    df_predictions_test=df_predictions_test_v0,   # output of the V0 one-shot run (grid_search_arimax_order)
    df_full=df_leg_current,
    horizons_h=(2, 6, 12, 24, 48),
)

### 4.4 Rolling update (V2)

At each new 2 h step, the real observed consumption is fed back to the model before predicting the next step
(coefficients are not re-estimated).

In [ ]:
def run_v0_v2_pleine_duree(leg_num=30):
    """
    Table best_order + RMSE (absolute and relative) for V0 (one-shot) and V2 (rolling),
    over the WHOLE test period — train/validation/test protocol (80/10/10).
    """

    df_leg_current = df_final_all[df_final_all["Leg"] == leg_num]
    if len(df_leg_current) == 0:
        print(f"Leg {leg_num} — no data, skipped")
        return None

    (_, _, _, _, _, _, _, _,
     df_predictions_test, _, _, _, _, _, _) = grid_search_arimax_order(
        df_leg_current,
        p_values=[0,1,2], d_values=[0], q_values=[0,1,2],
        freq="2h", criterion="RMSE_no_scaler",
        min_obs=30, target_agg="mean", verbose=False,
        versions_to_run=["V0", "V2"],
        split_mode="train_val_test", train_size=0.8, val_size=0.10,
        h_steps_v2=1,
    )

    # Number of test points per source
    df_n_test = df_predictions_test.groupby("SourceId").size().rename("n_test").reset_index()

    # Mean real value per source (for the relative RMSE)
    df_mean_true = df_predictions_test.groupby("SourceId")["y_true"].mean().rename("mean_y_true").reset_index()

    df_tab = df_predictions_test.drop_duplicates(subset=["SourceId"])[
        ["SourceId", "SourceName", "best_order_V0", "RMSE_test_V0",
         "best_order_V2", "RMSE_test_V2"]
    ].reset_index(drop=True)

    df_tab = df_tab.merge(df_n_test, on="SourceId", how="left")
    df_tab = df_tab.merge(df_mean_true, on="SourceId", how="left")

    df_tab["RMSE_test_V0"] = df_tab["RMSE_test_V0"].round(2)
    df_tab["RMSE_test_V2"] = df_tab["RMSE_test_V2"].round(2)

    # Relative RMSE (%) = RMSE / mean real value * 100
    df_tab["RMSE_rel_V0_%"] = (df_tab["RMSE_test_V0"] / df_tab["mean_y_true"] * 100).round(2)
    df_tab["RMSE_rel_V2_%"] = (df_tab["RMSE_test_V2"] / df_tab["mean_y_true"] * 100).round(2)

    df_tab = df_tab.drop(columns=["mean_y_true"])
    df_tab = df_tab.rename(columns={
        "n_test": "duree_test_h_approx",
        "RMSE_test_V0": "RMSE_V0",
        "RMSE_test_V2": "RMSE_V2",
    })
    df_tab["duree_test_h_approx"] = df_tab["duree_test_h_approx"] * 2

    print("\n" + "=" * 100)
    print(f"V0 vs V2 — WHOLE TEST PERIOD — Leg {leg_num} — train/validation/test (80/10/10)")
    print("=" * 100)
    print(df_tab.to_string(index=False))

    print(f"\nMean RMSE (all sources) — Leg {leg_num} :")
    print(f"  V0 : {df_tab['RMSE_V0'].mean():.2f}  (mean relative: {df_tab['RMSE_rel_V0_%'].mean():.2f} %)")
    print(f"  V2 : {df_tab['RMSE_V2'].mean():.2f}  (mean relative: {df_tab['RMSE_rel_V2_%'].mean():.2f} %)")

    # ---------------- FIGURES ----------------
    print(f"\nFigures — V0 (one-shot) vs V2 (rolling) — whole test period — Leg {leg_num}")
    plot_predictions(
        df_predictions_test=df_predictions_test,
        df_full=df_leg_current,
        split_mode="train_val_test",
        freq="2h",
        target_agg="mean",
        y_pred_v0_col="y_pred_v0",
        y_pred_v2_col="y_pred_v2",
    )

    return df_tab
resultats_pleine_duree_leg30 = run_v0_v2_pleine_duree(leg_num=30)


V0 vs V2 — WHOLE TEST PERIOD — Leg 30 — train/validation/test (80/10/10)
 SourceId                                             SourceName best_order_V0  RMSE_V0 best_order_V2  RMSE_V2  duree_test_h_approx  RMSE_rel_V0_%  RMSE_rel_V2_%
      103  RSB1_Switchboard.RSB1-S2_Reefer_RP7_-_Bay_22_Deck_Std     (2, 0, 0)     3.43     (2, 0, 2)     2.05                   60          19.17          11.46
      116  RSB2_Switchboard.RSB2-S4_Reefer_RP10_-_Bay_22_Deck_PS     (2, 0, 2)     4.30     (2, 0, 0)     2.51                   60          38.06          22.22
      126 RSB3_Switchboard.RSB3-S4_Reefer_RP21_-_Bay_46_deck_std     (1, 0, 1)     4.24     (2, 0, 1)     2.54                   60          33.15          19.86
      127 RSB3_Switchboard.RSB3-S4_Reefer_RP25_-_Bay_50_deck_Std     (2, 0, 1)     3.99     (2, 0, 2)     2.51                   60           7.98           5.02
      128 RSB3_Switchboard.RSB3-S4_Reefer_RP29_-_Bay_54_Deck_Std     (1, 0, 1)     2.85     (1, 0, 1)     2.63      

Comparison of V2 with a naive persistence baseline $\hat{y}(t) = y(t - 2h)$, to check that V2 is not simply copying the last observation.

In [ ]:
import plotly.graph_objects as go

# ============================================================
# 1. V2 predictions for all sources of Leg 30
# ============================================================

(_, _, _, _, _, _, _, _,
 df_predictions_test_v2, _, _, _, _, _, _) = grid_search_arimax_order(
    df_leg_current,
    p_values=[0, 1, 2], d_values=[0], q_values=[0, 1, 2],
    freq="2h", criterion="RMSE_no_scaler",
    min_obs=30, target_agg="mean", verbose=False,
    versions_to_run=["V2"],
    split_mode="train_val_test", train_size=0.8, val_size=0.10,
    h_steps_v2=1,   # update every 2 h with the real observed value
)

# ============================================================
# 2. "Persistence" baseline: y_naive(t) = y_real(t - 2h)
# ============================================================

def build_naive_series(df_full, source_id, leg_id=None,
                        timestamp_col="TimestampUTC", source_col="SourceId",
                        leg_col="Leg", target_col="Value",
                        freq="2h", target_agg="mean"):
    """Continuous series rebuilt (same processing as grid_search_arimax_order),
    then shifted by one step -> value at the previous step (t - freq)."""
    full_source = df_full[df_full[source_col].astype(str) == str(source_id)].copy()
    if leg_col in full_source.columns and leg_id is not None:
        full_source = full_source[full_source[leg_col].astype(str) == str(leg_id)]
    full_source[timestamp_col] = pd.to_datetime(full_source[timestamp_col])

    full_source = (
        full_source.groupby(timestamp_col, as_index=False)
        .agg({target_col: target_agg})
        .sort_values(timestamp_col)
        .set_index(timestamp_col)
    )
    if freq is not None:
        full_source = (
            full_source.resample(freq).agg({target_col: target_agg})
            .interpolate(method="time").dropna(subset=[target_col])
        )
    series = full_source[target_col]
    return series.shift(1)  # value at t-2h


# ============================================================
# 3. RMSE comparison V2 vs persistence, + figures per source
# ============================================================

rows_rmse = []
df_predictions_test_v2["TimestampUTC"] = pd.to_datetime(df_predictions_test_v2["TimestampUTC"])

for source_id, group in df_predictions_test_v2.groupby("SourceId"):
    group = group.sort_values("TimestampUTC").reset_index(drop=True)

    source_name = group["SourceName"].iloc[0] if "SourceName" in group.columns else ""
    leg_id      = group["Leg"].iloc[0] if "Leg" in group.columns else None

    # --- persistence baseline aligned on the test timestamps ---
    naive_series = build_naive_series(df_leg_current, source_id, leg_id)
    group["y_pred_naive"] = group["TimestampUTC"].map(naive_series.to_dict())

    valid = group.dropna(subset=["y_true", "y_pred_v2", "y_pred_naive"])
    if valid.empty:
        continue

    rmse_v2    = np.sqrt(mean_squared_error(valid["y_true"], valid["y_pred_v2"]))
    rmse_naive = np.sqrt(mean_squared_error(valid["y_true"], valid["y_pred_naive"]))
    rows_rmse.append({
        "SourceId": source_id, "SourceName": source_name,
        "RMSE_V2": round(rmse_v2, 2), "RMSE_naive": round(rmse_naive, 2),
        "V2_meilleur_que_naive": rmse_v2 < rmse_naive,
    })

    # --- figure: real / V2 / persistence ---
    title = f"V2 vs persistence (y(t) = y(t-2h)) — SourceId {source_id}"
    if source_name != "":
        title += f" | {source_name}"

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=group["TimestampUTC"], y=group["y_true"],
                              mode="lines", name="Real consumption",
                              line=dict(color="deepskyblue", width=2)))
    fig.add_trace(go.Scatter(x=group["TimestampUTC"], y=group["y_pred_v2"],
                              mode="lines", name="V2 (rolling ARIMAX)",
                              line=dict(color="steelblue", width=1.5)))
    fig.add_trace(go.Scatter(x=group["TimestampUTC"], y=group["y_pred_naive"],
                              mode="lines", name="Persistence (y(t-2h))",
                              line=dict(color="red", width=1.5, dash="dot")))

    fig.update_layout(title=title, xaxis_title="Time", yaxis_title="Consumption",
                       template="plotly_white", hovermode="x unified", width=1100, height=500)
    fig.show()

# ============================================================
# 4. Summary table
# ============================================================

df_rmse_compare = pd.DataFrame(rows_rmse)
print("\n=== V2 vs persistence baseline (y(t) = y(t-2h)) — all sources, Leg 30 ===")
print(df_rmse_compare.to_string(index=False))


=== V2 vs persistence baseline (y(t) = y(t-2h)) — all sources, Leg 30 ===
 SourceId                                             SourceName  RMSE_V2  RMSE_naive  V2_meilleur_que_naive
      103  RSB1_Switchboard.RSB1-S2_Reefer_RP7_-_Bay_22_Deck_Std     2.05        1.53                  False
      116  RSB2_Switchboard.RSB2-S4_Reefer_RP10_-_Bay_22_Deck_PS     2.51        2.67                   True
      126 RSB3_Switchboard.RSB3-S4_Reefer_RP21_-_Bay_46_deck_std     2.54        1.95                  False
      127 RSB3_Switchboard.RSB3-S4_Reefer_RP25_-_Bay_50_deck_Std     2.51        2.12                  False
      128 RSB3_Switchboard.RSB3-S4_Reefer_RP29_-_Bay_54_Deck_Std     2.63        3.07                   True
      129  RSB4_Switchboard.RSB4-S1_Reefer_RP16_-_Bay_38_Deck_PS     3.01        3.21                   True
      131  RSB4_Switchboard.RSB4-S2_Reefer_RP22_-_Bay_46_Deck_PS     2.73        2.62                  False
      132  RSB4_Switchboard.RSB4-S2_Reefer_RP26_-_Bay

### 4.5 Coefficient analysis (AR/MA root cancellation)

Explains why the rolling update has no effect on some sources (AR and MA roots nearly cancel out).

In [ ]:
(df_best_results_leg30, df_all_grid_leg30, df_all_predictions_leg30,
 best_models_leg30, df_horizon_results_leg30, df_horizon_preds_leg30,
 df_v2_horizon_results_leg30, df_v2_horizon_preds_leg30,
 df_predictions_test_leg30, *_ ) = grid_search_arimax_order(
    df_final_all[df_final_all["Leg"] == 30],
    p_values=[0,1,2], d_values=[0], q_values=[0,1,2],
    freq="2h", criterion="RMSE_no_scaler",
    min_obs=30, target_agg="mean", verbose=False,
    versions_to_run=["V0", "V2"],
    split_mode="train_val_test", train_size=0.8, val_size=0.10,
    h_steps_v2=1,
)
print(f"{len(best_models_leg30)} sources processed for Leg 30 — 80/10/10 protocol.")

NameError: name 'grid_search_arimax_order' is not defined

In [ ]:
import pandas as pd
import numpy as np

def tableau_comparatif_coeffs_v0_v2(best_models_leg30, df_v0_v2_table=None):
    lignes = []
    for sid, models in best_models_leg30.items():
        fit_v0 = models.get("no_scaler")
        fit_v2 = models.get("rolling")
        if fit_v0 is None or fit_v2 is None:
            continue

        def extraire(fit):
            params = pd.Series(fit.params, index=fit.param_names)
            ar_names = [n for n in params.index if n.startswith("ar.")]
            ma_names = [n for n in params.index if n.startswith("ma.")]
            exog_names = [n for n in params.index if n not in ar_names + ma_names + ["sigma2"]]
            beta = params[exog_names[0]] if exog_names else np.nan
            sum_ar = params[ar_names].sum() if ar_names else 0.0
            sum_ma_abs = params[ma_names].abs().sum() if ma_names else 0.0
            return fit.model.order, beta, sum_ar, sum_ma_abs

        order_v0, beta_v0, sum_ar_v0, sum_ma_v0 = extraire(fit_v0)
        order_v2, beta_v2, sum_ar_v2, sum_ma_v2 = extraire(fit_v2)

        lignes.append({
            "SourceId": sid,
            "order_V0": order_v0,
            "order_V2": order_v2,
            "beta_V0": round(beta_v0, 4),
            "beta_V2": round(beta_v2, 4),
            "sum_AR_V0": round(sum_ar_v0, 4),
            "sum_AR_V2": round(sum_ar_v2, 4),
            "sum_MA_abs_V0": round(sum_ma_v0, 4),
            "sum_MA_abs_V2": round(sum_ma_v2, 4),
        })

    df = pd.DataFrame(lignes).sort_values("SourceId").reset_index(drop=True)

    if df_v0_v2_table is not None:
        df = df.merge(
            df_v0_v2_table[["SourceId", "RMSE_rel_V0_%", "RMSE_rel_V2_%"]],
            on="SourceId", how="left"
        )
        df["gain_%"] = ((df["RMSE_rel_V0_%"] - df["RMSE_rel_V2_%"])
                         / df["RMSE_rel_V0_%"] * 100).round(1)

    return df


df_comparatif = tableau_comparatif_coeffs_v0_v2(best_models_leg30)
print(df_comparatif.to_string(index=False))

In [ ]:
def extraire_signes_ar_ma(best_models_leg30, sources_a_verifier):
    for sid in sources_a_verifier:
        for version, key in [("V0", "no_scaler"), ("V2", "rolling")]:
            fit = best_models_leg30[sid].get(key)
            if fit is None:
                continue
            params = pd.Series(fit.params, index=fit.param_names)
            ar_names = [n for n in params.index if n.startswith("ar.")]
            ma_names = [n for n in params.index if n.startswith("ma.")]
            print(f"Source {sid} ({version}) — AR: {params[ar_names].to_dict()}, "
                  f"MA: {params[ma_names].to_dict()}")

extraire_signes_ar_ma(best_models_leg30, [129, 131, 150, 142, 128])

## 5. XGBoost

A single **pooled** model is trained on all sources. Hyperparameters (`learning_rate`, `max_depth`, `n_estimators`)
are selected by grid search on the validation set, with early stopping.

### 5.1 Feature engineering

- Consumption history: value at t−1 and rolling mean / min / max / std over a window (4 h to 12 h, selected on validation)
- **Setpoint** (5 classes): fraction of reefers per class, thermal lift, setpoint heterogeneity (normalised entropy)
- **Equipment age** (4 classes): mean age, dispersion, fraction per class, heterogeneity

In [ ]:
"""
TNTM — Feature engineering (temperature windows, consumption lags, setpoint, age)
=========================================================

"""
def dedup_source_timestamp(df, source_col='SourceId', timestamp_col='TimestampUTC'):
    return df.drop_duplicates(subset=[source_col, timestamp_col]).reset_index(drop=True)

FENETRES_HEURES = {'4h': 2, '6h': 3, '8h': 4, '10h': 5, '12h': 6}


def build_temp_only_features(df, source_col='SourceId', timestamp_col='TimestampUTC',
                              temp_col='AIR_TEMPERATURE_2M', leg_col='Leg',
                              fenetres=FENETRES_HEURES):
    df = df.sort_values([source_col, leg_col, timestamp_col]).copy()

    group_keys = [source_col, leg_col]

    df['temp_lag1'] = df.groupby(group_keys)[temp_col].shift(1)

    df['_past_temp'] = df.groupby(group_keys)[temp_col].shift(1)
    roll = df.groupby(group_keys)['_past_temp']

    for label, window in fenetres.items():
        df[f'temp_mean_{label}'] = roll.transform(lambda s, w=window: s.rolling(w).mean())
        df[f'temp_min_{label}']  = roll.transform(lambda s, w=window: s.rolling(w).min())
        df[f'temp_max_{label}']  = roll.transform(lambda s, w=window: s.rolling(w).max())
        var = roll.transform(lambda s, w=window: s.rolling(w).var())
        df[f'temp_std_{label}'] = np.sqrt(var.clip(lower=0))

    df = df.drop(columns=['_past_temp'])

    return df


def add_conso_lag_features(df, source_col='SourceId', timestamp_col='TimestampUTC',
                            target_col='Value', leg_col='Leg',
                            fenetres=FENETRES_HEURES):
    df = df.sort_values([source_col, leg_col, timestamp_col]).copy()

    group_keys = [source_col, leg_col]

    df['conso_lag1'] = df.groupby(group_keys)[target_col].shift(1)

    df['_past_conso'] = df.groupby(group_keys)[target_col].shift(1)
    roll = df.groupby(group_keys)['_past_conso']

    for label, window in fenetres.items():
        df[f'conso_mean_{label}'] = roll.transform(lambda s, w=window: s.rolling(w).mean())
        df[f'conso_min_{label}']  = roll.transform(lambda s, w=window: s.rolling(w).min())
        df[f'conso_max_{label}']  = roll.transform(lambda s, w=window: s.rolling(w).max())
        var = roll.transform(lambda s, w=window: s.rolling(w).var())
        df[f'conso_std_{label}'] = np.sqrt(var.clip(lower=0))

    df = df.drop(columns=['_past_conso'])

    return df

# ======================================================================
#  Setpoint features (5 classes) + thermal lift
# ======================================================================

def setpoint_class(sp):
   
    if pd.isna(sp):
        return "unknown"
    if sp <= -28:
        return "deep_frozen"
    elif -28 < sp <= -18:
        return "frozen"
    elif -18 < sp <= 0:
        return "chilled"
    elif 0 < sp <= 10:
        return "fresh"
    elif 10 < sp <= 28:
        return "ambient"
    else:
        return "unknown"   


def add_setpoint_features(df, setpoint_col='TEMPERATURE_CARRIAGE',
                           source_col='SourceId', timestamp_col='TimestampUTC',
                           temp_col='AIR_TEMPERATURE_2M'):
    """
    Weighted fraction of reefers per setpoint class (f_k,s,t) + setpoint_heterogeneity,
    and source_thermal_lift = weighted sum of max(0, T_amb - T_set,k).
    """
    df = df.copy()
    df['setpoint_class'] = df[setpoint_col].apply(setpoint_class)

    T_SET = {
        'deep_frozen': -28,
        'frozen': (-28 + -18) / 2,
        'chilled': (-18 + 0) / 2,
        'fresh': (0 + 10) / 2,
        'ambient': (10 + 28) / 2,
    }
    K = len(T_SET)

    def _agg(g):
        g_valid = g[g['setpoint_class'] != 'unknown'] # setpoint available (not NaN)
        n_total = len(g_valid) # number of valid reefers at this time on this source
        result = {}
        if n_total == 0:
            for k in T_SET:
                result[f'fraction_setpoint_{k}'] = np.nan
            result['source_thermal_lift'] = np.nan
            result['setpoint_heterogeneity'] = np.nan
            return pd.Series(result)

        temp_ext = g[temp_col].iloc[0]
        thermal_lift = 0.0
        entropy = 0.0
        for k, T_k in T_SET.items(): # k = class name, T_k = its representative temperature
            n_k = (g_valid['setpoint_class'] == k).sum() # number of reefers in class k
            f_k = n_k / n_total 
            result[f'fraction_setpoint_{k}'] = f_k
            thermal_lift += f_k * max(0, temp_ext - T_k)
            if f_k > 0:
                entropy += f_k * np.log(f_k)

        result['source_thermal_lift'] = thermal_lift
        result['setpoint_heterogeneity'] = -entropy / np.log(K)
        return pd.Series(result)

    agg = df.groupby([source_col, timestamp_col]).apply(_agg).reset_index()
    return df.merge(agg, on=[source_col, timestamp_col], how='left')




# ======================================================================
# G5 -- Age (imputation + 4 classes)
# ======================================================================

def impute_age(df, age_col='EQP_AGE', version='min'):
    df = df.copy()
    if version == 'min':
        fill_val = df[age_col].min()
    elif version == 'max':
        fill_val = df[age_col].max()
    else:
        raise ValueError("version must be 'min' or 'max'")
    df[f'{age_col}_imputed'] = df[age_col].fillna(fill_val)
    return df


def age_class(age):
   
    if pd.isna(age):
        return 'unknown'
    if age < 5:
        return 'young'
    elif age < 10:          
        return 'moderate'
    elif age < 15:          
        return 'old'
    else:                     
        return 'very_old'


def add_age_features(df, age_col='EQP_AGE_imputed', source_col='SourceId', timestamp_col='TimestampUTC'):
    
    df = df.copy()
    df['age_class'] = df[age_col].apply(age_class)
    classes = ['young', 'moderate', 'old', 'very_old']
    K = len(classes)

    def _agg(g):
        valid = g[age_col].dropna()
        n = len(valid) # number of reefers with a valid age 
        result = {}
        if n == 0:
            result['mean_reefer_age'] = np.nan
            result['age_dispersion'] = np.nan
            for c in classes:
                result[f'fract_age_{c}'] = np.nan
            result['age_heterogeneity'] = np.nan
            return pd.Series(result)

        mean_age = valid.mean()
        disp = np.sqrt(((valid - mean_age) ** 2).mean())  # .mean() divides by the total number of reefers
        result['mean_reefer_age'] = mean_age
        result['age_dispersion'] = disp

        entropy = 0.0
        for c in classes:
            n_c = (g['age_class'] == c).sum() # number of reefers of the group belonging to class c
            f_c = n_c / n # n = total number of reefers with a valid age
            result[f'fract_age_{c}'] = f_c
            if f_c > 0:
                entropy += f_c * np.log(f_c)
        result['age_heterogeneity'] = -entropy / np.log(K)
        return pd.Series(result)

    agg = df.groupby([source_col, timestamp_col]).apply(_agg).reset_index()
    return df.merge(agg, on=[source_col, timestamp_col], how='left')


# ======================================================================
# FULL PIPELINE -- builds ALL the features 
# ======================================================================

def build_all_features(df_leg, age_version='min', include_conso_lags=True, fenetres=FENETRES_HEURES):
    df = df_leg.copy()

    df = add_setpoint_features(df)
    df = impute_age(df, version=age_version)
    df = add_age_features(df)

    df = dedup_source_timestamp(df)

    df = build_temp_only_features(df, fenetres=fenetres)
    if include_conso_lags:
        df = add_conso_lag_features(df, fenetres=fenetres)

    return df
# ======================================================================
# CELLULE 1 -- FEATURES
# ======================================================================

# ======================================================================
# BUILD df_features: Leg 30 only + 7 legs
# ======================================================================

# --- Variant A: Leg 30 only ---
LEGS_LEG30_SEUL = [30]
df_legs_leg30 = df_final_all[df_final_all["Leg"].isin(LEGS_LEG30_SEUL)].copy()
df_features_leg30 = build_all_features(
    df_legs_leg30, age_version='min', include_conso_lags=True
)
print(f"df_features_leg30 : {len(df_features_leg30)} rows, {df_features_leg30['SourceId'].nunique()} sources")

# --- Variant B: all 7 legs ---
LEGS_TOUS = [21, 22, 23, 24, 30, 34, 44]
df_legs_tous = df_final_all[df_final_all["Leg"].isin(LEGS_TOUS)].copy()
df_features_tous = build_all_features(
    df_legs_tous, age_version='min', include_conso_lags=True
)
print(f"df_features_tous : {len(df_features_tous)} rows, {df_features_tous['SourceId'].nunique()} sources")

### 5.2 Pooled training with hyperparameter grid search

In [ ]:
import itertools

BEST_HYPERPARAMS = {'learning_rate': 0.3, 'max_depth': 6, 'n_estimators': 100}

HYPERPARAM_GRID = {
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'max_depth': [3, 4, 6],
    'n_estimators': [50, 100, 200],
}


def run_xgboost_pooled(df_features, features, feature_set_name,
                        target='Value', source_col='SourceId', timestamp_col='TimestampUTC',
                        leg_col='Leg',
                        train_size=0.8, val_size=0.10,
                        hyperparam_grid=HYPERPARAM_GRID,
                        early_stopping_rounds=20,
                        verbose_grid=False):
    """
    POOLED XGBoost with hyperparameter grid search, same principle as
    grid_search_arimax_order:
      1. chronological train/val/test split per (source, leg)
      2. for each hyperparameter combination: train on train,
         evaluate the RMSE on val
      3. keep the combination with the best val RMSE
      4. evaluate this frozen model on test
    """
    df = df_features.sort_values([source_col, leg_col, timestamp_col]).copy()
    parts = []
    for (sid, leg), g in df.groupby([source_col, leg_col]):
        g = g.sort_values(timestamp_col).reset_index(drop=True)
        n = len(g)
        n_train = int(n * train_size)
        n_val = int(n * val_size)
        g['split'] = 'test'
        g.loc[:n_train - 1, 'split'] = 'train'
        g.loc[n_train:n_train + n_val - 1, 'split'] = 'val'
        parts.append(g)
    df = pd.concat(parts, ignore_index=True)

    df_clean = df.dropna(subset=features + [target]).copy()
    train = df_clean[df_clean['split'] == 'train']
    val = df_clean[df_clean['split'] == 'val']
    test = df_clean[df_clean['split'] == 'test']

    if len(train) == 0 or len(val) == 0 or len(test) == 0:
        nan_counts = df[df['split'] == 'test'][features + [target]].isna().sum()
        raise ValueError(
            f"[{feature_set_name}] Empty split after dropna: "
            f"train={len(train)}, val={len(val)}, test={len(test)}.\n"
            f"NaN per column on the 'test' rows (before dropna):\n{nan_counts}"
        )

    # --- 1. Grid search over all hyperparameter combinations ---
    keys = list(hyperparam_grid.keys())
    combinaisons = list(itertools.product(*hyperparam_grid.values()))

    best_rmse_val = np.inf
    best_hyperparams = None
    best_model = None

    for combo in combinaisons:
        hp = dict(zip(keys, combo))

        model_candidate = XGBRegressor(
            **hp,
            objective='reg:squarederror',
            early_stopping_rounds=early_stopping_rounds,
            eval_metric='rmse'
        )
        model_candidate.fit(train[features], train[target],
                             eval_set=[(val[features], val[target])], verbose=False)

        y_pred_val = model_candidate.predict(val[features])
        rmse_val = np.sqrt(mean_squared_error(val[target], y_pred_val))

        if verbose_grid:
            print(f"  hp={hp} -> RMSE val={rmse_val:.3f}")

        if rmse_val < best_rmse_val:
            best_rmse_val = rmse_val
            best_hyperparams = hp
            best_model = model_candidate

    print(f"[{feature_set_name}] Best hyperparameters (val RMSE): {best_hyperparams} "
          f"(RMSE val={best_rmse_val:.3f})")

    # --- 2. Evaluation of the frozen best model on test ---
    model = best_model
    y_pred_test = model.predict(test[features])
    rmse_global = np.sqrt(mean_squared_error(test[target], y_pred_test))

    results = []
    test_with_pred = test.copy()
    test_with_pred['predicted'] = y_pred_test
    for (sid, leg), g_test in test_with_pred.groupby([source_col, leg_col]):
        rmse_source = np.sqrt(mean_squared_error(g_test[target], g_test['predicted']))
        results.append({'feature_set': feature_set_name, 'SourceId': sid, 'Leg': leg,
                          'n_test': len(g_test), 'RMSE': rmse_source})
    df_results = pd.DataFrame(results)
    df_results.attrs['rmse_global'] = rmse_global
    df_results.attrs['best_iteration'] = model.best_iteration
    df_results.attrs['best_hyperparams'] = best_hyperparams
    df_results.attrs['best_rmse_val'] = best_rmse_val

    y_pred_all = model.predict(df_clean[features].ffill())
    df_predictions = df_clean[[source_col, leg_col, timestamp_col, target, 'split']].copy()
    df_predictions['predicted'] = y_pred_all
    df_predictions['feature_set'] = feature_set_name

    return df_results, df_predictions, model

In [ ]:
def run_xgboost_window_grid_conso(df_features, fenetres, features_fixes, feature_set_prefix,
                                  hyperparam_grid=HYPERPARAM_GRID, verbose=True):
    """
    For each window: builds the feature set (features_fixes + conso_lag1 +
    conso_mean/min/max/std_<window>) and runs run_xgboost_pooled (hyperparameter
    grid search on validation). Then keeps the BEST WINDOW according to the
    VALIDATION RMSE (best_rmse_val), never the test set.
    """
    resultats_par_fenetre = {}

    for label, window in fenetres.items():
        features = features_fixes + ['conso_lag1', f'conso_mean_{label}', f'conso_min_{label}',
                                      f'conso_max_{label}', f'conso_std_{label}']
        set_name = f'{feature_set_prefix}_{label}'

        df_res, df_pred, model = run_xgboost_pooled(
            df_features, features=features, feature_set_name=set_name,
            hyperparam_grid=hyperparam_grid, verbose_grid=False
        )
        resultats_par_fenetre[label] = {
            'df_results': df_res, 'df_pred': df_pred, 'model': model,
            'rmse_val': df_res.attrs['best_rmse_val'],
            'hyperparams': df_res.attrs['best_hyperparams'],
        }
        if verbose:
            print(f"  [{set_name}] window={label} -> RMSE val={df_res.attrs['best_rmse_val']:.3f}, "
                  f"RMSE test={df_res.attrs['rmse_global']:.3f}, hp={df_res.attrs['best_hyperparams']}")

    meilleure_fenetre = min(resultats_par_fenetre, key=lambda l: resultats_par_fenetre[l]['rmse_val'])
    best = resultats_par_fenetre[meilleure_fenetre]

    print(f"\n>>> Selected window for '{feature_set_prefix}' : {meilleure_fenetre} "
          f"(RMSE val={best['rmse_val']:.3f}, RMSE test={best['df_results'].attrs['rmse_global']:.3f}, "
          f"hyperparams={best['hyperparams']})")

    return best['df_results'], best['df_pred'], best['model'], meilleure_fenetre

In [ ]:
# ======================================================================
# RELATIVE RMSE HELPERS -- mean real consumption over the test window
# ======================================================================

def get_conso_moyenne_test(df_full, df_pred_or_test, target_col='Value',
                            source_col='SourceId', leg_col='Leg',
                            timestamp_col='TimestampUTC', has_split_col=True):
    """
    Computes, for each (source, leg), the mean real consumption over the
    test window.

    - If df_pred_or_test has a 'split' column (XGBoost): uses the timestamps
      where split == 'test'.
    - Otherwise (ARIMAX, df_pred_or_test already contains test rows only):
      uses the min/max available timestamps as test window and reads the real
      consumption from df_full over this window.
    """
    df_pred_or_test = df_pred_or_test.copy()
    df_pred_or_test[timestamp_col] = pd.to_datetime(df_pred_or_test[timestamp_col])
    df_full = df_full.copy()
    df_full[timestamp_col] = pd.to_datetime(df_full[timestamp_col])

    if has_split_col and 'split' in df_pred_or_test.columns:
        test_rows = df_pred_or_test[df_pred_or_test['split'] == 'test']
    else:
        test_rows = df_pred_or_test

    fenetres = test_rows.groupby([source_col, leg_col])[timestamp_col].agg(['min', 'max']).reset_index()
    fenetres = fenetres.rename(columns={'min': 't_start', 'max': 't_end'})

    resultats = []
    for _, row in fenetres.iterrows():
        sid, leg = row[source_col], row[leg_col]
        mask = (
            (df_full[source_col] == sid) &
            (df_full[leg_col] == leg) &
            (df_full[timestamp_col] >= row['t_start']) &
            (df_full[timestamp_col] <= row['t_end'])
        )
        conso_moy = df_full.loc[mask, target_col].mean()
        resultats.append({source_col: sid, leg_col: leg, 'conso_moy_test': conso_moy})

    return pd.DataFrame(resultats)


def add_rmse_relatif(df_results, conso_moyennes, source_col='SourceId', leg_col='Leg'):
    """Adds RMSE_relatif = RMSE / conso_moy_test to a df_results."""
    df_results = df_results.merge(conso_moyennes, on=[source_col, leg_col], how='left')
    df_results['RMSE_relatif'] = df_results['RMSE'] / df_results['conso_moy_test']
    return df_results

In [ ]:
def plot_predictions_comparaison(df_pred_a, df_pred_b, df_full, label_a, label_b,
                                  title_suffix="", timestamp_col="TimestampUTC",
                                  source_col="SourceId", source_name_col="SourceName",
                                  leg_col="Leg", target_col="Value", freq="2h",
                                  target_agg="mean"):
    """
    Per source: real consumption, config A (solid green), config B
    (dotted orange). Generic plot to compare two feature sets.
    """
    df_a_test = df_pred_a[df_pred_a['split'] == 'test'].copy()
    df_a_test[timestamp_col] = pd.to_datetime(df_a_test[timestamp_col])

    df_b_test = df_pred_b[df_pred_b['split'] == 'test'].copy()
    df_b_test[timestamp_col] = pd.to_datetime(df_b_test[timestamp_col])

    df_full_plot = df_full.copy()
    df_full_plot[timestamp_col] = pd.to_datetime(df_full_plot[timestamp_col])

    for (source_id, leg_id), group_a in df_a_test.groupby([source_col, leg_col]):
        group_a = group_a.sort_values(timestamp_col)
        group_b = df_b_test[(df_b_test[source_col] == source_id) &
                             (df_b_test[leg_col] == leg_id)].sort_values(timestamp_col)

        source_name = group_a[source_name_col].iloc[0] if source_name_col in group_a.columns else ""
        current_split_time = group_a[timestamp_col].min()

        title = f"Prediction - SourceId {source_id}"
        if source_name != "":
            title += f" | {source_name}"
        title += f" | Leg {leg_id}" + title_suffix

        fig = go.Figure()

        full_source = df_full_plot[df_full_plot[source_col].astype(str) == str(source_id)].copy()
        if leg_col in full_source.columns:
            full_source = full_source[full_source[leg_col].astype(str) == str(leg_id)]
        if not full_source.empty:
            full_source = (full_source.groupby(timestamp_col, as_index=False)
                        .agg({target_col: target_agg}).sort_values(timestamp_col)
                        .set_index(timestamp_col))
            if freq is not None:
                full_source = (full_source.resample(freq).agg({target_col: target_agg})
                            .interpolate(method="time").dropna(subset=[target_col]))
            full_source = full_source.reset_index()
            fig.add_trace(go.Scatter(x=full_source[timestamp_col], y=full_source[target_col],
                                    mode="lines", name="Real consumption",
                                    line=dict(color="deepskyblue", width=2)))

        fig.add_trace(go.Scatter(x=group_a[timestamp_col], y=group_a['predicted'],
                                mode="lines", name=label_a,
                                line=dict(color="green", width=1.5)))

        if not group_b.empty:
            fig.add_trace(go.Scatter(x=group_b[timestamp_col], y=group_b['predicted'],
                                    mode="lines", name=label_b,
                                    line=dict(color="darkorange", width=1.5, dash="dot")))

        fig.add_shape(type="line", x0=current_split_time, x1=current_split_time, y0=0, y1=1,
                      xref="x", yref="paper")
        fig.add_annotation(x=current_split_time, y=1, xref="x", yref="paper",
                            text="Validation/Test split", showarrow=False, yanchor="bottom")

        fig.update_layout(title=title, xaxis_title="Time", yaxis_title="Consumption",
                        template="plotly_white", hovermode="x unified", width=1150, height=550)
        fig.show()

### 5.3 ARIMAX vs XGBoost at equivalent memory

To compare both models fairly, XGBoost gets the same memory as ARIMAX's autoregressive part:
- $p = 0$: ambient temperature only
- $p = 1$: temperature + consumption at $t-1$
- $p = 2$: temperature + consumption at $t-1$ and $t-2$

For each source, $p$ is selected on the **validation** RMSE (same protocol as the ARIMAX order), and the test RMSE of that $p$ is reported.

In [ ]:
df_leg_current = df_final_all[df_final_all["Leg"] == 30].copy()

In [ ]:
# ======================================================================
# DYNAMIC XGBoost SYSTEM -- equivalent to ARIMAX memory (p=0,1,2), one-step-ahead
# p selected on VALIDATION (as for ARIMAX), never on test
# For each source: keep the optimal p, one curve per source.
# Leg 30 only AND 7 legs, raw + relative RMSE.
# ======================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

def add_ar_lag_features(df, source_col='SourceId', timestamp_col='TimestampUTC',
                         target_col='Value', leg_col='Leg', n_lags=2):
    df = df.sort_values([source_col, leg_col, timestamp_col]).copy()
    group_keys = [source_col, leg_col]
    lag_cols = []
    for lag in range(1, n_lags + 1):
        col_name = f'conso_lag{lag}'
        df[col_name] = df.groupby(group_keys)[target_col].shift(lag)
        lag_cols.append(col_name)
    return df, lag_cols


FEATURES_EXOGENES = ['AIR_TEMPERATURE_2M']

FEATURES_PAR_ORDRE = {
    0: FEATURES_EXOGENES,
    1: FEATURES_EXOGENES + ['conso_lag1'],
    2: FEATURES_EXOGENES + ['conso_lag1', 'conso_lag2'],
}


def run_arimax_v0_only(leg_num=30):
    df_leg_current = df_final_all[df_final_all["Leg"] == leg_num]
    if len(df_leg_current) == 0:
        return None
    (_, _, _, _, _, _, _, _,
     df_predictions_test, _, _, _, _, _, _) = grid_search_arimax_order(
        df_leg_current,
        p_values=[0, 1, 2], d_values=[0], q_values=[0, 1, 2],
        freq="2h", criterion="RMSE_no_scaler",
        min_obs=30, target_agg="mean", verbose=False,
        versions_to_run=["V0"],
        split_mode="train_val_test", train_size=0.8, val_size=0.10,
        h_steps_v2=1,
    )
    return df_predictions_test


def rmse_par_source(df_pred, split_name, target='Value',
                     source_col='SourceId', leg_col='Leg'):
    """RMSE per (source, leg), on the requested split ('val' or 'test')."""
    df_split = df_pred[df_pred['split'] == split_name]
    rows = []
    for (sid, leg), g in df_split.groupby([source_col, leg_col]):
        rmse = np.sqrt(mean_squared_error(g[target], g['predicted']))
        rows.append({source_col: sid, leg_col: leg, 'RMSE': rmse})
    return pd.DataFrame(rows)


def run_ar_system_best_p(df_features, label_perimetre):
    """
    Trains XGBoost for p=0,1,2. For EACH SOURCE, selects the p that minimises
    the VALIDATION RMSE (never the test -- as for ARIMAX), then reports the test
    RMSE of this p. Plots ONE curve per source with this optimal p.
    """
    print(f"\n{'#'*70}\nDYNAMIC AR SYSTEM -- {label_perimetre}\n{'#'*70}")

    df_features_local, _ = add_ar_lag_features(df_features, n_lags=2)

    resultats_p = {}
    predictions_p = {}
    rmse_val_p = {}
    rmse_test_p = {}

    for p, features in FEATURES_PAR_ORDRE.items():
        set_name = f'xgb_ar_p{p}_{label_perimetre}'
        df_res, df_pred, model = run_xgboost_pooled(
            df_features_local, features=features, feature_set_name=set_name
        )
        resultats_p[p] = df_res
        predictions_p[p] = df_pred
        rmse_val_p[p] = rmse_par_source(df_pred, 'val').rename(columns={'RMSE': f'RMSE_val_p{p}'})
        rmse_test_p[p] = rmse_par_source(df_pred, 'test').rename(columns={'RMSE': f'RMSE_test_p{p}'})
        print(f"p={p} -- RMSE global test : {df_res.attrs['rmse_global']:.3f}")

    # --- Merge val + test for the 3 orders ---
    df_val_p = rmse_val_p[0][['SourceId', 'Leg', 'RMSE_val_p0']]
    df_test_p = rmse_test_p[0][['SourceId', 'Leg', 'RMSE_test_p0']]
    for p in [1, 2]:
        df_val_p = df_val_p.merge(rmse_val_p[p][['SourceId', 'Leg', f'RMSE_val_p{p}']], on=['SourceId', 'Leg'])
        df_test_p = df_test_p.merge(rmse_test_p[p][['SourceId', 'Leg', f'RMSE_test_p{p}']], on=['SourceId', 'Leg'])

    df_compare_p = df_val_p.merge(df_test_p, on=['SourceId', 'Leg'])

    # --- Selection of p on VALIDATION only ---
    val_cols = ['RMSE_val_p0', 'RMSE_val_p1', 'RMSE_val_p2']
    df_compare_p = df_compare_p.dropna(subset=val_cols, how='all').copy()
    df_compare_p['best_p'] = (
        df_compare_p[val_cols].idxmin(axis=1)
        .str.replace('RMSE_val_p', '', regex=False).astype(int)
    )

    def get_rmse_test(row):
        p = int(row['best_p'])
        col = f'RMSE_test_p{p}'
        return row[col] if col in row.index and pd.notna(row[col]) else np.nan

    df_compare_p['RMSE_best_p'] = df_compare_p.apply(get_rmse_test, axis=1)

    df_compare_p_leg30 = df_compare_p[df_compare_p['Leg'] == 30].sort_values('SourceId')

    print(f"\n=== TABLE -- val/test RMSE per source, Leg 30, optimal p selected on VAL ({label_perimetre}) ===")
    cols_aff = ['SourceId', 'Leg'] + val_cols + [f'RMSE_test_p{p}' for p in [0, 1, 2]] + ['best_p', 'RMSE_best_p']
    print(df_compare_p_leg30[cols_aff].round(2).to_string(index=False))
    print(f"\nDistribution of the best p ({label_perimetre}) :")
    print(df_compare_p['best_p'].value_counts().sort_index())

    # --- Build a single df_pred, keeping for each source only its best p ---
    morceaux = []
    for _, row in df_compare_p_leg30.iterrows():
        sid, best_p = row['SourceId'], int(row['best_p'])
        g = predictions_p[best_p][
            (predictions_p[best_p]['SourceId'] == sid) & (predictions_p[best_p]['Leg'] == 30)
        ].copy()
        g['best_p_utilise'] = best_p
        morceaux.append(g)
    df_pred_best_leg30 = pd.concat(morceaux, ignore_index=True)

    return df_compare_p_leg30, df_pred_best_leg30


def plot_predictions_ar_best_p(df_pred, df_full, df_arimax_v0=None, title_suffix="",
                                timestamp_col="TimestampUTC", source_col="SourceId",
                                source_name_col="SourceName", leg_col="Leg",
                                target_col="Value", freq="2h", target_agg="mean"):
    """Real consumption + ARIMAX V0 + XGBoost (optimal p per source), one figure per source."""
    df_test = df_pred[df_pred['split'] == 'test'].copy()
    df_test[timestamp_col] = pd.to_datetime(df_test[timestamp_col])
    df_full_plot = df_full.copy()
    df_full_plot[timestamp_col] = pd.to_datetime(df_full_plot[timestamp_col])
    if df_arimax_v0 is not None:
        df_arimax_v0 = df_arimax_v0.copy()
        df_arimax_v0[timestamp_col] = pd.to_datetime(df_arimax_v0[timestamp_col])

    for (source_id, leg_id), group in df_test.groupby([source_col, leg_col]):
        group = group.sort_values(timestamp_col)
        source_name = group[source_name_col].iloc[0] if source_name_col in group.columns else ""
        best_p_used = group['best_p_utilise'].iloc[0] if 'best_p_utilise' in group.columns else "?"
        current_split_time = group[timestamp_col].min()

        title = f"Prediction - SourceId {source_id}"
        if source_name != "":
            title += f" | {source_name}"
        title += f" | Leg {leg_id} | p={best_p_used}" + title_suffix

        fig = go.Figure()
        full_source = df_full_plot[df_full_plot[source_col].astype(str) == str(source_id)].copy()
        if leg_col in full_source.columns:
            full_source = full_source[full_source[leg_col].astype(str) == str(leg_id)]
        if not full_source.empty:
            full_source = (full_source.groupby(timestamp_col, as_index=False)
                        .agg({target_col: target_agg}).sort_values(timestamp_col)
                        .set_index(timestamp_col))
            if freq is not None:
                full_source = (full_source.resample(freq).agg({target_col: target_agg})
                            .interpolate(method="time").dropna(subset=[target_col]))
            full_source = full_source.reset_index()
            fig.add_trace(go.Scatter(x=full_source[timestamp_col], y=full_source[target_col],
                                    mode="lines", name="Real consumption",
                                    line=dict(color="deepskyblue", width=2)))

        if df_arimax_v0 is not None:
            group_arimax = df_arimax_v0[df_arimax_v0[source_col] == source_id].sort_values(timestamp_col)
            if not group_arimax.empty and 'y_pred_v0' in group_arimax.columns:
                fig.add_trace(go.Scatter(x=group_arimax[timestamp_col], y=group_arimax['y_pred_v0'],
                                        mode="lines", name="ARIMAX V0", line=dict(color="red", width=1.5)))

        fig.add_trace(go.Scatter(x=group[timestamp_col], y=group['predicted'],
                                mode="lines", name=f"XGBoost (p={best_p_used})",
                                line=dict(color="green", width=1.5)))

        fig.add_shape(type="line", x0=current_split_time, x1=current_split_time, y0=0, y1=1,
                      xref="x", yref="paper")
        fig.add_annotation(x=current_split_time, y=1, xref="x", yref="paper",
                            text="Validation/Test split", showarrow=False, yanchor="bottom")
        fig.update_layout(title=title, xaxis_title="Time", yaxis_title="Consumption",
                        template="plotly_white", hovermode="x unified", width=1150, height=550)
        fig.show()


# --- ARIMAX V0, Leg 30 only (common reference) ---
df_arimax_v0 = run_arimax_v0_only(leg_num=30)

# --- VERSION A: training on Leg 30 only ---
df_compare_p_leg30_A, df_pred_best_leg30_A = run_ar_system_best_p(
    df_features_leg30, label_perimetre='Leg 30 only'
)
plot_predictions_ar_best_p(
    df_pred_best_leg30_A, df_leg_current, df_arimax_v0,
    title_suffix=" (AR optimal p per source, VAL — trained on Leg 30 only)"
)

# --- VERSION B: training on 7 legs ---
df_compare_p_leg30_B, df_pred_best_leg30_B = run_ar_system_best_p(
    df_features_tous, label_perimetre='7 legs'
)
plot_predictions_ar_best_p(
    df_pred_best_leg30_B, df_leg_current, df_arimax_v0,
    title_suffix=" (AR optimal p per source, VAL — trained on 7 legs)"
)

# --- Final comparison: ARIMAX vs XGBoost (optimal p on VAL), + relative RMSE ---
if df_arimax_v0 is not None:
    df_arimax_best = df_arimax_v0.drop_duplicates(subset=['SourceId'])[
        ['SourceId', 'SourceName', 'best_order_V0', 'RMSE_test_V0']
    ].rename(columns={'RMSE_test_V0': 'RMSE_ARIMAX', 'best_order_V0': 'best_order_ARIMAX'})

    conso_moy_arimax = get_conso_moyenne_test(df_leg_current, df_arimax_v0, has_split_col=False)
    df_arimax_best = df_arimax_best.merge(conso_moy_arimax, on='SourceId', how='left')
    df_arimax_best['RMSE_relatif_ARIMAX'] = df_arimax_best['RMSE_ARIMAX'] / df_arimax_best['conso_moy_test']

    for df_compare_p_leg30, label in [(df_compare_p_leg30_A, 'Leg 30 only'), (df_compare_p_leg30_B, '7 legs')]:
        df_final_compare = df_arimax_best.merge(
            df_compare_p_leg30[['SourceId', 'best_p', 'RMSE_best_p']], on='SourceId'
        )
        df_final_compare['RMSE_relatif_XGBoost'] = (
            df_final_compare['RMSE_best_p'] / df_final_compare['conso_moy_test']
        )
        df_final_compare['gain_XGBoost_vs_ARIMAX_%'] = (
            (df_final_compare['RMSE_ARIMAX'] - df_final_compare['RMSE_best_p'])
            / df_final_compare['RMSE_ARIMAX'] * 100
        ).round(1)

        print(f"\n=== COMPARISON -- XGBoost (optimal p, VAL) vs ARIMAX, training on {label} ===")
        cols = ['SourceId', 'SourceName', 'best_order_ARIMAX', 'RMSE_ARIMAX', 'RMSE_relatif_ARIMAX',
                'best_p', 'RMSE_best_p', 'RMSE_relatif_XGBoost', 'gain_XGBoost_vs_ARIMAX_%']
        print(df_final_compare[cols].round(3).to_string(index=False))
        print(f"Mean relative RMSE -- ARIMAX : {df_final_compare['RMSE_relatif_ARIMAX'].mean()*100:.1f} %"
              f"  |  XGBoost ({label}) : {df_final_compare['RMSE_relatif_XGBoost'].mean()*100:.1f} %")

### 5.4 Enrichment with contextual features (pooled training on 7 legs, evaluation on Leg 30)

Each configuration is evaluated in two settings: training on Leg 30 only, and pooled training on 7 legs.

#### Baseline: ambient temperature + consumption history

In [ ]:
# ======================================================================
# GRID SEARCH WINDOWS + HYPERPARAMETERS -- TEMPERATURE (instantaneous, no
# lags) + CONSUMPTION LAGS only
# VERSION A: training on Leg 30 only, prediction on Leg 30
# VERSION B: training on 7 legs, prediction on Leg 30
# ======================================================================

def plot_predictions_simple(df_pred, df_full,
                             timestamp_col="TimestampUTC", source_col="SourceId",
                             source_name_col="SourceName", leg_col="Leg",
                             target_col="Value", freq="2h", target_agg="mean",
                             title_suffix=""):
    """
    Per source: real consumption, XGBoost (solid green).
    """
    df_test = df_pred[df_pred['split'] == 'test'].copy()
    df_test[timestamp_col] = pd.to_datetime(df_test[timestamp_col])

    df_full_plot = df_full.copy()
    df_full_plot[timestamp_col] = pd.to_datetime(df_full_plot[timestamp_col])

    for (source_id, leg_id), group in df_test.groupby([source_col, leg_col]):
        group = group.sort_values(timestamp_col)
        source_name = group[source_name_col].iloc[0] if source_name_col in group.columns else ""
        current_split_time = group[timestamp_col].min()

        title = f"Prediction - SourceId {source_id}"
        if source_name != "":
            title += f" | {source_name}"
        title += f" | Leg {leg_id}"
        title += title_suffix

        fig = go.Figure()

        full_source = df_full_plot[df_full_plot[source_col].astype(str) == str(source_id)].copy()
        if leg_col in full_source.columns:
            full_source = full_source[full_source[leg_col].astype(str) == str(leg_id)]
        if not full_source.empty:
            full_source = (full_source.groupby(timestamp_col, as_index=False)
                        .agg({target_col: target_agg}).sort_values(timestamp_col)
                        .set_index(timestamp_col))
            if freq is not None:
                full_source = (full_source.resample(freq).agg({target_col: target_agg})
                            .interpolate(method="time").dropna(subset=[target_col]))
            full_source = full_source.reset_index()
            fig.add_trace(go.Scatter(x=full_source[timestamp_col], y=full_source[target_col],
                                    mode="lines", name="Real consumption",
                                    line=dict(color="deepskyblue", width=2)))

        fig.add_trace(go.Scatter(x=group[timestamp_col], y=group['predicted'],
                                mode="lines", name="XGBoost",
                                line=dict(color="green", width=1.5)))

        fig.add_shape(type="line", x0=current_split_time, x1=current_split_time, y0=0, y1=1,
                      xref="x", yref="paper")
        fig.add_annotation(x=current_split_time, y=1, xref="x", yref="paper",
                            text="Validation/Test split", showarrow=False, yanchor="bottom")

        fig.update_layout(title=title, xaxis_title="Time", yaxis_title="Consumption",
                        template="plotly_white", hovermode="x unified", width=1150, height=550)
        fig.show()


def run_conso_lags(df_features, label_perimetre, df_full):
    """
    Runs TEMPERATURE (instantaneous) + CONSUMPTION LAGS on the given
    df_features (grid search over windows + hyperparameters). Prints the
    raw + relative RMSE table (Leg 30) and plots the curves.
    """
    print(f"\n{'#'*70}\nTRAINING PERIMETER: {label_perimetre}\n{'#'*70}")

    print(f"\n{'='*70}\nGRID SEARCH WINDOWS -- temp + conso_lags ({label_perimetre})\n{'='*70}")
    df_res, df_pred, model, fenetre = run_xgboost_window_grid_conso(
        df_features, FENETRES_HEURES,
        features_fixes=['AIR_TEMPERATURE_2M'],
        feature_set_prefix=f'temp_conso_lags_{label_perimetre}'
    )

    # --- Filter on Leg 30 (evaluation always on Leg 30) ---
    df_res_leg30 = df_res[df_res['Leg'] == 30].copy()
    df_pred_leg30 = df_pred[df_pred['Leg'] == 30]

    # --- Relative RMSE ---
    conso_moy = get_conso_moyenne_test(df_full, df_pred_leg30, has_split_col=True)
    df_res_leg30 = add_rmse_relatif(df_res_leg30, conso_moy)

    print(f"\n{'='*100}\nRMSE TABLE raw + relative -- Leg 30, training {label_perimetre}, "
          f"temp + conso_lags (window={fenetre})\n{'='*100}")
    print(df_res_leg30[['SourceId', 'RMSE', 'RMSE_relatif']].round(3).to_string(index=False))

    print(f"\nMean RMSE_relatif (all sources) -- {label_perimetre} :")
    print(f"  XGBoost : {df_res_leg30['RMSE_relatif'].mean()*100:.1f} %")

    plot_predictions_simple(
        df_pred=df_pred_leg30,
        df_full=df_full,
        title_suffix=f" (XGBoost temp+conso_lags {fenetre} — trained on {label_perimetre})"
    )

    return df_res_leg30


# --- VERSION A: training on Leg 30 only ---
df_tableau_conso_leg30 = run_conso_lags(
    df_features_leg30, label_perimetre='Leg 30 only', df_full=df_leg_current
)

# --- VERSION B: training on 7 legs ---
df_tableau_conso_7legs = run_conso_lags(
    df_features_tous, label_perimetre='7 legs', df_full=df_leg_current
)

#### + Setpoint features

In [ ]:
# ======================================================================
# RELATIVE RMSE + FIGURES -- "temp + conso_lags" vs
# "setpoint + temp + conso_lags", Leg 30 only and 7 legs
# ======================================================================

FEATURES_TEMP_SEULE = ['AIR_TEMPERATURE_2M']

FEATURES_SETPOINT = [
    'AIR_TEMPERATURE_2M',
    'fraction_setpoint_deep_frozen', 'fraction_setpoint_frozen',
    'fraction_setpoint_chilled', 'fraction_setpoint_fresh',
    'fraction_setpoint_ambient', 'source_thermal_lift',
    'setpoint_heterogeneity'
]


def comparer_temp_vs_setpoint(df_features, label_perimetre, df_full):
    """
    Runs both configurations (temp+conso_lags, setpoint+temp+conso_lags) on the
    same df_features/perimeter, builds a comparison table (raw + relative RMSE
    per source) and plots the curves.
    """
    df_temp, df_pred_temp, model_temp, fenetre_temp = run_xgboost_window_grid_conso(
        df_features, FENETRES_HEURES,
        features_fixes=FEATURES_TEMP_SEULE,
        feature_set_prefix=f'temp_conso_lags_{label_perimetre}'
    )
    df_setpoint, df_pred_setpoint, model_setpoint, fenetre_setpoint = run_xgboost_window_grid_conso(
        df_features, FENETRES_HEURES,
        features_fixes=FEATURES_SETPOINT,
        feature_set_prefix=f'setpoint_conso_lags_{label_perimetre}'
    )

    df_temp_leg30 = df_temp[df_temp['Leg'] == 30].copy()
    df_setpoint_leg30 = df_setpoint[df_setpoint['Leg'] == 30].copy()
    df_pred_temp_leg30 = df_pred_temp[df_pred_temp['Leg'] == 30]
    df_pred_setpoint_leg30 = df_pred_setpoint[df_pred_setpoint['Leg'] == 30]

    conso_moy_temp = get_conso_moyenne_test(df_full, df_pred_temp_leg30, has_split_col=True)
    conso_moy_setpoint = get_conso_moyenne_test(df_full, df_pred_setpoint_leg30, has_split_col=True)

    df_temp_leg30 = add_rmse_relatif(df_temp_leg30, conso_moy_temp)
    df_setpoint_leg30 = add_rmse_relatif(df_setpoint_leg30, conso_moy_setpoint)

    df_compare = df_temp_leg30[['SourceId', 'RMSE', 'RMSE_relatif']].rename(
        columns={'RMSE': 'RMSE_temp', 'RMSE_relatif': 'RMSE_relatif_temp'}
    ).merge(
        df_setpoint_leg30[['SourceId', 'RMSE', 'RMSE_relatif']].rename(
            columns={'RMSE': 'RMSE_setpoint', 'RMSE_relatif': 'RMSE_relatif_setpoint'}
        ), on='SourceId'
    )

    df_compare['gain_setpoint_%'] = (
        (df_compare['RMSE_temp'] - df_compare['RMSE_setpoint'])
        / df_compare['RMSE_temp'] * 100
    ).round(1)

    print(f"\n{'='*100}")
    print(f"COMPARISON TABLE -- temp+conso_lags vs setpoint+conso_lags ({label_perimetre}) "
          f"(window temp={fenetre_temp}, window setpoint={fenetre_setpoint})")
    print(f"{'='*100}")
    print(df_compare.round(3).to_string(index=False))
    print(f"\nMean relative RMSE -- temp+conso_lags     : {df_compare['RMSE_relatif_temp'].mean()*100:.1f} %")
    print(f"Mean relative RMSE -- setpoint+conso_lags : {df_compare['RMSE_relatif_setpoint'].mean()*100:.1f} %")

    plot_predictions_comparaison(
        df_pred_a=df_pred_temp_leg30,
        df_pred_b=df_pred_setpoint_leg30,
        df_full=df_full,
        label_a=f"temp+conso_lags ({fenetre_temp})",
        label_b=f"setpoint+temp+conso_lags ({fenetre_setpoint})",
        title_suffix=f" (temp vs setpoint+temp, {label_perimetre})"
    )

    return df_compare


# --- VERSION A: training on Leg 30 only ---
df_compare_leg30 = comparer_temp_vs_setpoint(
    df_features_leg30, label_perimetre='Leg 30 only', df_full=df_leg_current
)

# --- VERSION B: training on 7 legs ---
df_compare_7legs = comparer_temp_vs_setpoint(
    df_features_tous, label_perimetre='7 legs', df_full=df_leg_current
)

#### + Equipment age features (missing ages imputed with the minimum value)

In [ ]:
# ======================================================================
# RELATIVE RMSE + FIGURES -- "temp + conso_lags" vs
# "age + temp + conso_lags", Leg 30 only and 7 legs
# ======================================================================

FEATURES_AGE = [
    'AIR_TEMPERATURE_2M',
    'mean_reefer_age', 'age_dispersion', 'fract_age_young',
    'fract_age_moderate', 'fract_age_old', 'fract_age_very_old',
    'age_heterogeneity'
]


def comparer_temp_vs_age(df_features, label_perimetre, df_full):
    """
    Runs both configurations (temp+conso_lags, age+temp+conso_lags) on the
    same df_features/perimeter, builds a comparison table (raw + relative RMSE
    per source) and plots the curves.
    """
    df_temp, df_pred_temp, model_temp, fenetre_temp = run_xgboost_window_grid_conso(
        df_features, FENETRES_HEURES,
        features_fixes=FEATURES_TEMP_SEULE,
        feature_set_prefix=f'temp_conso_lags_{label_perimetre}'
    )
    df_age, df_pred_age, model_age, fenetre_age = run_xgboost_window_grid_conso(
        df_features, FENETRES_HEURES,
        features_fixes=FEATURES_AGE,
        feature_set_prefix=f'age_conso_lags_{label_perimetre}'
    )

    df_temp_leg30 = df_temp[df_temp['Leg'] == 30].copy()
    df_age_leg30 = df_age[df_age['Leg'] == 30].copy()
    df_pred_temp_leg30 = df_pred_temp[df_pred_temp['Leg'] == 30]
    df_pred_age_leg30 = df_pred_age[df_pred_age['Leg'] == 30]

    conso_moy_temp = get_conso_moyenne_test(df_full, df_pred_temp_leg30, has_split_col=True)
    conso_moy_age = get_conso_moyenne_test(df_full, df_pred_age_leg30, has_split_col=True)

    df_temp_leg30 = add_rmse_relatif(df_temp_leg30, conso_moy_temp)
    df_age_leg30 = add_rmse_relatif(df_age_leg30, conso_moy_age)

    df_compare = df_temp_leg30[['SourceId', 'RMSE', 'RMSE_relatif']].rename(
        columns={'RMSE': 'RMSE_temp', 'RMSE_relatif': 'RMSE_relatif_temp'}
    ).merge(
        df_age_leg30[['SourceId', 'RMSE', 'RMSE_relatif']].rename(
            columns={'RMSE': 'RMSE_age', 'RMSE_relatif': 'RMSE_relatif_age'}
        ), on='SourceId'
    )

    df_compare['gain_age_%'] = (
        (df_compare['RMSE_temp'] - df_compare['RMSE_age'])
        / df_compare['RMSE_temp'] * 100
    ).round(1)

    print(f"\n{'='*100}")
    print(f"COMPARISON TABLE -- temp+conso_lags vs age+temp+conso_lags ({label_perimetre}) "
          f"(window temp={fenetre_temp}, window age={fenetre_age})")
    print(f"{'='*100}")
    print(df_compare.round(3).to_string(index=False))
    print(f"\nMean relative RMSE -- temp+conso_lags      : {df_compare['RMSE_relatif_temp'].mean()*100:.1f} %")
    print(f"Mean relative RMSE -- age+temp+conso_lags  : {df_compare['RMSE_relatif_age'].mean()*100:.1f} %")

    plot_predictions_comparaison(
        df_pred_a=df_pred_temp_leg30,
        df_pred_b=df_pred_age_leg30,
        df_full=df_full,
        label_a=f"temp+conso_lags ({fenetre_temp})",
        label_b=f"age+temp+conso_lags ({fenetre_age})",
        title_suffix=f" (temp vs age+temp, {label_perimetre})"
    )

    return df_compare


# --- VERSION A: training on Leg 30 only ---
df_compare_age_leg30 = comparer_temp_vs_age(
    df_features_leg30, label_perimetre='Leg 30 only', df_full=df_leg_current
)

# --- VERSION B: training on 7 legs ---
df_compare_age_7legs = comparer_temp_vs_age(
    df_features_tous, label_perimetre='7 legs', df_full=df_leg_current
)

#### + Equipment age features (missing ages imputed with the maximum value — retained)

In [ ]:
# ======================================================================
# FEATURE CONSTRUCTION -- missing ages imputed with the MAXIMUM value
# ======================================================================

# --- Variant A: Leg 30 only, age imputation = max ---
df_features_leg30_max = build_all_features(
    df_legs_leg30, age_version='max', include_conso_lags=True
)
print(f"df_features_leg30_max : {len(df_features_leg30_max)} rows, "
      f"{df_features_leg30_max['SourceId'].nunique()} sources")

# --- Variant B: all 7 legs, age imputation = max ---
df_features_tous_max = build_all_features(
    df_legs_tous, age_version='max', include_conso_lags=True
)
print(f"df_features_tous_max : {len(df_features_tous_max)} rows, "
      f"{df_features_tous_max['SourceId'].nunique()} sources")


# ======================================================================
# RELATIVE RMSE + FIGURES -- temp+conso_lags vs age+temp+conso_lags
# age imputation = MAX, Leg 30 only and 7 legs
# ======================================================================

# --- VERSION A: training on Leg 30 only, age imputation max ---
df_compare_age_max_leg30 = comparer_temp_vs_age(
    df_features_leg30_max, label_perimetre='Leg 30 only (age max)', df_full=df_leg_current
)

# --- VERSION B: training on 7 legs, age imputation max ---
df_compare_age_max_7legs = comparer_temp_vs_age(
    df_features_tous_max, label_perimetre='7 legs (age max)', df_full=df_leg_current
)

### 5.5 Summary of the enrichment

In [ ]:
# ======================================================================
# RELATIVE RMSE (w.r.t. the REAL CONSUMPTION) -- 3 versions:
# 1. temp + conso_lags
# 2. temp + conso_lags + setpoint
# 3. temp + conso_lags + age (imputation max)
# ======================================================================

def run_3_versions_rmse_relatif(df_features, df_features_age_max,
                                label_perimetre, df_full):
    """
    Runs the 3 versions and computes, for each one, the relative RMSE per
    source = RMSE / mean REAL consumption over the test period. It measures
    directly the gap between each version and the real consumption -- it is
    NOT a gain w.r.t. a reference version.
    """
    FEATURES_SETPOINT = [
        'AIR_TEMPERATURE_2M',
        'fraction_setpoint_deep_frozen', 'fraction_setpoint_frozen',
        'fraction_setpoint_chilled', 'fraction_setpoint_fresh',
        'fraction_setpoint_ambient', 'source_thermal_lift',
        'setpoint_heterogeneity'
    ]
    FEATURES_AGE = [
        'AIR_TEMPERATURE_2M',
        'mean_reefer_age', 'age_dispersion', 'fract_age_young',
        'fract_age_moderate', 'fract_age_old', 'fract_age_very_old',
        'age_heterogeneity'
    ]

    configs = {
        'temp_conso_lags':          (df_features,         FEATURES_TEMP_SEULE),
        'temp_conso_lags_setpoint': (df_features,         FEATURES_SETPOINT),
        'temp_conso_lags_age_max':  (df_features_age_max, FEATURES_AGE),
    }

    resultats = {}

    for nom_config, (df_feat, features_fixes) in configs.items():
        print(f"\n{'#'*70}\n{nom_config.upper()} -- {label_perimetre}\n{'#'*70}")

        df_res, df_pred, model, fenetre = run_xgboost_window_grid_conso(
            df_feat, FENETRES_HEURES,
            features_fixes=features_fixes,
            feature_set_prefix=f'{nom_config}_{label_perimetre}'
        )

        df_res_leg30 = df_res[df_res['Leg'] == 30].copy()
        df_pred_leg30 = df_pred[df_pred['Leg'] == 30]

        # relative RMSE = RMSE / mean REAL consumption over the test period
        conso_moy = get_conso_moyenne_test(df_full, df_pred_leg30, has_split_col=True)
        df_res_leg30 = add_rmse_relatif(df_res_leg30, conso_moy)

        resultats[nom_config] = {
            'df_res': df_res_leg30[['SourceId', 'RMSE', 'RMSE_relatif']],
            'fenetre': fenetre
        }
        print(f"Selected window : {fenetre} | Mean relative RMSE (vs real consumption) : "
              f"{df_res_leg30['RMSE_relatif'].mean()*100:.2f} %")

    # --- Final table: relative RMSE of each version, per source ---
    df_synth = resultats['temp_conso_lags']['df_res'][['SourceId']].copy()

    for nom_config in configs.keys():
        df_synth = df_synth.merge(
            resultats[nom_config]['df_res'][['SourceId', 'RMSE_relatif']].rename(
                columns={'RMSE_relatif': f'RMSE_rel_{nom_config}'}
            ), on='SourceId'
        )

    print(f"\n{'='*100}")
    print(f"TABLE -- relative RMSE w.r.t. real consumption, {label_perimetre}")
    print(f"{'='*100}")
    print(df_synth.round(4).to_string(index=False))

    print(f"\n--- Mean relative RMSE per version (w.r.t. real consumption), {label_perimetre} ---")
    for nom_config in configs.keys():
        print(f"  {nom_config:30s} : {df_synth[f'RMSE_rel_{nom_config}'].mean()*100:.2f} %")

    return df_synth


# --- VERSION A: training on Leg 30 only ---
df_synth_leg30 = run_3_versions_rmse_relatif(
    df_features_leg30, df_features_leg30_max,
    label_perimetre='Leg 30 only', df_full=df_leg_current
)

# --- VERSION B: training on 7 legs ---
df_synth_7legs = run_3_versions_rmse_relatif(
    df_features_tous, df_features_tous_max,
    label_perimetre='7 legs', df_full=df_leg_current
)